# Notebook 03: Monthly & Annual LST Temporal Analysis

## Overview
This notebook extracts and analyzes Land Surface Temperature (LST) data from MODIS Terra (MOD11A1) and Aqua (MYD11A1) satellites using Google Earth Engine. It processes all cities with delineated AOIs from **Notebook 01** and performs comprehensive temporal trend analysis.

## Prerequisites
-  Completed **Notebook 01** (AOI Delineation)
-  Configured `config.py` with valid GEE credentials
-  Activated virtual environment with required packages

## Key Capabilities
- **Multi-City Processing**: Automatically detects and processes all AOI outputs
- **Temporal Scope**: 2006-2025 (configurable)
- **LST Acquisition**: Day, Night, or Both (set in `config.py`)
- **Quality Assurance**: Hierarchical QA filtering (Strict â†’ Moderate)
- **Parallel Processing**: Dask-based parallelization for efficiency
- **Statistical Analysis**: Mann-Kendall trend tests, linear regression, climatology
- **Stakeholder Validation**: Compare perceptions with measured trends

## Workflow Structure
1. **Setup & Authentication** â†’ Initialize GEE and Dask
2. **Load Configuration**  †’ Import settings from `config.py`
3. **Load AOI Data**  †’ Auto-load city boundaries
4. **LST Extraction**  †’ Download MODIS LST with QA filtering
5. **Visualize LST Maps**  †’ Generate monthly/annual maps
6. **Compute Statistics**  †’ Calculate temporal metrics
7. **Trend Analysis**  †’ Perform statistical tests
8. **Cross-City Comparison** â†’ Create heatmaps
9. **Stakeholder Validation** â†’ Compare perceptions vs. reality

## Output Directories
```
data/outputs/03_trend_analysis_monthly_annual/
â”œâ”€â”€ City_*/
â”‚   â””â”€”€ LST_Temporal_Analysis/
”‚       ”œ”€”€ Day/ (or Night/)
”‚       ”‚   ”œ”€”€ GeoTIFF/
”‚       ”‚   ”œ”€”€ Figures/
”‚       ”‚   ”””€”€ Statistics/
”‚       ”””€”€ Cross_City_Statistics/
```

## Next Steps
After completing this notebook, proceed to **Notebook 04** for daily LST analysis and diurnal temperature variations.

## Mathematical Background: Trend Analysis

### 1. Linear Regression Model
For each city and time period (day/night), we fit an ordinary least squares (OLS) regression:

$$\text{LST}_t = \beta_0 + \beta_1 \cdot t + \epsilon_t$$

where:
- $\text{LST}_t$ = median LST at time $t$ (°C)
- $t$ = year (2006, 2007, ..., 2025)
- $\beta_0$ = intercept (baseline temperature)
- $\beta_1$ = slope (°C/year)
- $\epsilon_t$ = residual error

### 2. Warming/Cooling Rate Metrics

**Per Decade Rate:**
$$\Delta T_{decade} = 10 \cdot \beta_1$$

**Total Study Period Change:**
$$\Delta T_{total} = \beta_1 \cdot (t_{end} - t_{start})$$

For 2006-2025: $\Delta T_{total} = \beta_1 \cdot 19$

### 3. Statistical Significance

**Coefficient of Determination ($R^2$):**
$$R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

**P-value Interpretation:**
- $p < 0.05$ †’ Statistically significant trend (*)
- $p \geq 0.05$ †’ No significant trend

### 4. Mann-Kendall Trend Test
Non-parametric test for monotonic trends:

$$S = \sum_{i=1}^{n-1} \sum_{j=i+1}^{n} \text{sgn}(x_j - x_i)$$

Test statistic $Z$ follows standard normal distribution under null hypothesis (no trend).

## Block 1: Setup and Authentication

In [ ]:
# =============================================================================
# IMPORT REQUIRED LIBRARIES
# =============================================================================

# Core libraries
import os
import sys
import json
import time
import re
import glob
import calendar
import warnings
import traceback
import requests
import shutil
from datetime import datetime, timedelta

# Data processing
import numpy as np
import pandas as pd

# Geospatial libraries
import ee                             # Google Earth Engine API
import geemap                         # Interactive mapping with Earth Engine
import geopandas as gpd               # Geospatial data handling
import rasterio                       # Raster data I/O
from rasterio.plot import show        # Plotting raster data
import pyproj                         # Projections and transformations
# Ensure Point is imported for LST extraction
from shapely.geometry import Point

# Statistical analysis
from scipy import stats               # Statistical functions
from scipy.optimize import curve_fit  # Curve fitting
# Use np.inf for infinity, do not import inf from scipy

# Parallel processing with Dask
import dask
from dask.distributed import Client, progress, wait
from dask import delayed
import multiprocessing
from tqdm.notebook import tqdm        # Progress bars

# Visualization
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
from matplotlib.cm import ScalarMappable
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
import seaborn as sns

# Fix for libpysal import error with scipy.inf
# Monkey patch scipy.inf before libpysal is imported
import scipy
if not hasattr(scipy, 'inf'):
    import numpy as np
    scipy.inf = np.inf
    
# Spatial statistics (for hotspot analysis)
import libpysal
from libpysal.weights import Queen, KNN
from esda.getisord import G_Local
import pymannkendall

# Import project configuration and custom modules
sys.path.insert(0, os.path.abspath('../'))
import config
from src import gee_utils, temporal_analysis, visualization

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')

print("Libraries imported successfully")
print(f"PROJECT root: {config.PROJECT_ROOT}")
print(f"DATA directory: {config.DATA_DIR}")

# Configure backend-safe savefig behavior via modularized data_io utilities
from src import data_io

data_io.install_backend_safe_savefig(svg_mirror_pdf=True)
print('Installed backend-safe savefig patch from src.data_io (SVG mirrored to PDF)')

In [ ]:
# =============================================================================
# INITIALIZE GOOGLE EARTH ENGINE
# =============================================================================

print("="*70)
print("INITIALIZING GOOGLE EARTH ENGINE")
print("="*70)

# Initialize GEE using modularized function
gee_available = gee_utils.initialize_gee(
    config.GEE_CONFIG['service_json'],
    config.GEE_CONFIG['project_id']
)

if gee_available:
    print("Google Earth Engine initialized successfully")
    print(f"Project ID: {config.GEE_CONFIG['project_id']}")
else:
    print("Google Earth Engine initialization failed")
    print("Some features will be limited")

# Store GEE availability for later checks
GEE_AVAILABLE = gee_available

print("="*70)

In [ ]:
# Initialize Dask for parallel processing
# Determine number of workers based on CPU cores (leave 1 core for system operations)
n_workers = max(1, multiprocessing.cpu_count() - 1)
n_threads = 2  # Threads per worker

print(f"Setting up Dask with {n_workers} workers and {n_threads} threads per worker")

# Initialize Dask client for parallel processing
client = Client(n_workers=n_workers, threads_per_worker=n_threads)
print(f"Dask dashboard available at: {client.dashboard_link}")

# Function to print Dask cluster information
def print_dask_info():
    info = client.scheduler_info()
    print(f"Dask Cluster Information:")
    print(f"  Workers: {len(info['workers'])}")
    print(f"  Total Threads: {sum(w['nthreads'] for w in info['workers'].values())}")
    print(f"  Total Memory: {sum(w['memory_limit'] for w in info['workers'].values()) / 1e9:.2f} GB")

# Print Dask information
print_dask_info()

## Block 2: Load Configuration

All settings are centralized in [`config.py`](../config.py). This cell creates local references for backward compatibility.

**Key Configuration Parameters:**
- `LST_CONFIG['lst_time']` â†’ 'day', 'night', or 'both'
- `TEMPORAL_CONFIG` â†’ Date ranges (2006-2025), aggregation methods
- `PROCESSING_OPTIONS` â†’ QA tiers (Strict/Moderate), Dask workers
- `VISUALIZATION_CONFIG` â†’ Color schemes, temperature ranges

In [ ]:
# =============================================================================
# LOAD PROJECT CONFIGURATION
# =============================================================================

print("="*70)
print("LOADING PROJECT CONFIGURATION")
print("="*70)

# All configuration is now centralized in config.py
# Users should modify config.py to change behavior

# Create local references for backward compatibility with existing code
# These are REFERENCES, not copies - they point to config.py
PROJECT_CONFIG = {
    'study_year': str(config.PROJECT_SETTINGS['study_year']),
    'start_date': config.TEMPORAL_CONFIG['start_date'],
    'end_date': config.TEMPORAL_CONFIG['end_date'],
    'data_dir': str(config.DATA_DIR),
    'output_dir': str(config.OUTPUT_DATA_PATH),
    'aoi_results_dir': str(config.get_output_dir('01_aoi_delineation')),
    'lst_output_dir': str(config.get_output_dir('03_trend_analysis_monthly_annual')),
    'aoi_pattern': config.PROJECT_SETTINGS['aoi_pattern'],
    'daily_stats_subdir': config.PROJECT_SETTINGS['daily_stats_subdir'],
    'hot_pct': config.PROJECT_SETTINGS['hot_pct']
}

LST_CONFIG = config.LST_CONFIG
TEMPORAL_CONFIG = config.TEMPORAL_CONFIG
VISUALIZATION_CONFIG = config.VISUALIZATION_CONFIG
PROCESSING_OPTIONS = config.PROCESSING_OPTIONS

# Quick validation of LST time selection
valid_lst_times = ['day', 'night', 'both']
if LST_CONFIG['lst_time'] not in valid_lst_times:
    raise ValueError(f"Invalid lst_time in config.py: {LST_CONFIG['lst_time']}. Must be one of {valid_lst_times}")

print(f"Configuration loaded from config.py")
print(f"Time period: {PROJECT_CONFIG['start_date']} to {PROJECT_CONFIG['end_date']}")
print(f"LST time mode: {LST_CONFIG['lst_time']}")
print(f"QA levels to use: {LST_CONFIG['qa_levels_to_use']}")
print(f"Primary collection: {LST_CONFIG['primary_collection']}")
print(f"Processing options: {PROCESSING_OPTIONS['qa_filtering']} QA filtering")
print("\nNOTE: To modify settings, edit config.py and restart the script.")
print("="*70)

## Block 3: Load AOI Data

Auto-loads city boundaries from **Notebook 01** outputs:
- Reads AOI GeoJSON files from `data/outputs/01_aoi_delineation/City_*/`
- Extracts bounding boxes for LST extraction
- Validates geometry and CRS compatibility

In [ ]:
# =============================================================================
# LOAD AOI DATA
# =============================================================================

print("="*70)
print("LOADING AOI BOUNDARIES")
print("="*70)

# Use modularized function to load AOI boundaries
aoi_data = gee_utils.load_aoi_boundaries(
    aoi_results_dir=PROJECT_CONFIG['aoi_results_dir']
)

print(f"\nLoaded AOI data for {len(aoi_data)} cities:")
for city in sorted(aoi_data.keys()):
    bounds = aoi_data[city]['bounds']
    print(f"{city}: Bounds [{bounds[0]:.3f}, {bounds[1]:.3f}, {bounds[2]:.3f}, {bounds[3]:.3f}]")
    
print("="*70)

## Block 4: LST Extraction with Hierarchical QA Filtering

**Parallel Processing Strategy:**
- Dask workers process multiple cities simultaneously
- Dynamic resource allocation based on CPU cores
- Real-time progress tracking with `tqdm`

**Hierarchical Quality Assurance (QA):**

MODIS LST products include QA bands with error estimates:

| QA Level | Bits | Error Threshold | Description |
|----------|------|-----------------|-------------|
| **Strict** | 00 | â‰¤ 1K | Highest quality pixels |
| **Moderate** | 01 | â‰¤ 2K | Medium quality pixels |
| ~~Loose~~ | ~~10~~ | ~~â‰¤ 3K~~ | Not used in this study |

**Fallback Logic:**
1. Extract LST using Strict QA mask
2. If insufficient data â†’ Apply Moderate QA mask
3. Merge results with Strict taking precedence

**Output Products:**
- Monthly climatology GeoTIFFs (12 months)
- Yearly LST GeoTIFFs (2006-2025)
- QA composition maps (percentage of Strict/Moderate pixels)

**Processing Time:** ~5-15 minutes per city (depends on AOI size and time range)

In [ ]:
# Import load_modis_collection_with_hierarchical_qa from src
# (previously defined inline â€” now lives in src/gee_utils.py)
import importlib
import src.gee_utils
importlib.reload(src.gee_utils)
from src.gee_utils import load_modis_collection_with_hierarchical_qa

print("Imported load_modis_collection_with_hierarchical_qa from src.gee_utils")

In [ ]:
# Import compute_and_export_lst_geotiff_with_qa from src
# (previously defined inline â€” now lives in src/gee_utils.py)
from src.gee_utils import compute_and_export_lst_geotiff_with_qa

print("Imported compute_and_export_lst_geotiff_with_qa from src.gee_utils")

In [ ]:
# DASK-BASED GEE DOWNLOADS
# Functions init_ee_in_worker and process_city_period imported from src

import os
import time
from tqdm.notebook import tqdm
from dask.distributed import Client, as_completed
from dask import delayed

from src.gee_utils import init_ee_in_worker, process_city_period

# Main execution block
print("\nðŸš€ Starting Dask-based LST Data Processing")
print("="*80)

# First, ensure all workers have the EE initialization function
client.run(init_ee_in_worker)

# Create task list
tasks = []
lst_time_options = ['day', 'night'] if LST_CONFIG['lst_time'] == 'both' else [LST_CONFIG['lst_time']]

for city_name, aoi_info in aoi_data.items():
    # Convert bounds to list (not numpy array)
    bounds = aoi_info['bounds']
    bounds_list = bounds.tolist() if hasattr(bounds, 'tolist') else list(bounds)
    city_dir = aoi_info['city_dir']
    
    for lst_time in lst_time_options:
        # Yearly tasks
        for year in TEMPORAL_CONFIG['years']:
            tasks.append({
                'city_name': city_name,
                'bounds_list': bounds_list,
                'city_dir': city_dir,
                'period_type': 'yearly',
                'period_value': year,
                'lst_time': lst_time
            })
        
        # Monthly climatology tasks
        for month in TEMPORAL_CONFIG['months']:
            tasks.append({
                'city_name': city_name,
                'bounds_list': bounds_list,
                'city_dir': city_dir,
                'period_type': 'monthly_climatology',
                'period_value': month,
                'lst_time': lst_time
            })

print(f"Created {len(tasks)} tasks for parallel processing")
print(f"Cities: {', '.join(aoi_data.keys())}")
print(f"Years: {TEMPORAL_CONFIG['years'][0]} - {TEMPORAL_CONFIG['years'][-1]}")
print(f"LST Time: {', '.join(lst_time_options)}")

# Create Dask delayed tasks
dask_tasks = []
for task in tasks:
    dask_task = delayed(process_city_period)(
        task['city_name'],
        task['bounds_list'],
        task['city_dir'],
        task['period_type'],
        task['period_value'],
        task['lst_time'],
        dict(LST_CONFIG),
        dict(TEMPORAL_CONFIG),
        dict(PROJECT_CONFIG)
    )
    dask_tasks.append(dask_task)

# Submit tasks
print(f"\n Submitting tasks to Dask cluster...")
print(f"Workers: {len(client.scheduler_info()['workers'])}")
print(f"Dashboard: {client.dashboard_link}")
futures = client.compute(dask_tasks)

# Track progress
results = []
errors = []

with tqdm(total=len(tasks), desc="Processing LST data") as pbar:
    for future, result in as_completed(futures, with_results=True):
        results.append(result)
        
        if result['status'] == 'downloaded':
            pbar.set_postfix_str(f" {result['city']} - {result['period_type']}={result['period_value']} ({result['lst_time']})")
        elif result['status'] == 'skipped':
            pbar.set_postfix_str(f" Skipped: {result['city']}")
        elif result['status'] == 'error':
            errors.append(result)
            pbar.set_postfix_str(f"Error: {result['city']}")
            
        pbar.update(1)

# Summary
print("\n" + "="*80)
print(" PROCESSING SUMMARY")
print("="*80)

downloaded = [r for r in results if r['status'] == 'downloaded']
skipped = [r for r in results if r['status'] == 'skipped']
no_data = [r for r in results if r['status'] == 'no_data']

print(f"Total tasks: {len(results)}")
print(f"Downloaded: {len(downloaded)}")
print(f"Skipped (already exist): {len(skipped)}")
print(f"No data available: {len(no_data)}")
print(f"Errors: {len(errors)}")

if errors:
    print("\n ERRORS:")
    for error in errors[:5]:
        print(f"\n{error['city']} - {error['period_type']}={error['period_value']} ({error['lst_time']}):")
        print(f"  {error['message']}")

print("\n Processing complete!")
print(f"Check output directories in: {PROJECT_CONFIG['lst_output_dir']}")

## Block 5: Visualize LST Maps

Generates publication-ready maps in A5 format (5.83" Ã— 8.27"):

**Monthly Climatology Maps:**
- 12 subplots (one per month) in single figure
- Color scale: Perceptually uniform "RdYlBu_r"
- Consistent scale across all months for comparability

**Yearly LST Maps:**
- 20 subplots (2006-2025) in single figure
- Tracks inter-annual temperature variations
- Identifies anomalously warm/cool years

**QA Composition Maps:**
- Visualizes spatial distribution of QA levels
- Color coding: Green (Strict) â†’ Yellow (Moderate) â†’ Red (Poor/No Data)

**Output Formats:** SVG (screen viewing) and SVG (publication-ready)

In [ ]:
print("="*70)
print("DASK-BASED VISUALIZATION WITH QA STATISTICS")
print("="*70)

import os
import importlib
from dask import delayed
from dask.distributed import as_completed
from tqdm.notebook import tqdm

# Reload visualization module to pick up newly added functions
import src.visualization
importlib.reload(src.visualization)

# Import visualization functions from src (previously defined inline)
from src.visualization import (
    load_geotiff, calculate_qa_stats,
    create_monthly_climatology_map, create_yearly_lst_map
)

# --- Skip logic: check if all expected outputs already exist ---
_skip_viz = config.SKIP_EXISTING.get('lst_visualization', False)
_all_exist = False

if _skip_viz:
    _all_exist = True  # assume all exist, flip if any missing
    lst_time_options_check = ['day', 'night'] if LST_CONFIG['lst_time'] == 'both' else [LST_CONFIG['lst_time']]
    for _cn in aoi_data:
        for _lt in lst_time_options_check:
            _sub = 'Day' if _lt == 'day' else 'Night'
            _fdir = os.path.join(PROJECT_CONFIG['lst_output_dir'], f"City_{_cn}",
                                 'LST_Temporal_Analysis', _sub, 'Figures')
            # Check for at least the monthly climatology SVG
            _expected_monthly = os.path.join(_fdir, f"{_cn}_{_lt.title()}_Monthly_Climatology.svg")
            _expected_yearly = os.path.join(_fdir, f"{_cn}_{_lt.title()}_Yearly_LST.svg")
            if not os.path.exists(_expected_monthly) or not os.path.exists(_expected_yearly):
                _all_exist = False
                break
        if not _all_exist:
            break

if _skip_viz and _all_exist:
    print("  SKIPPED â€” all LST visualization outputs already exist.")
    print("   Set config.SKIP_EXISTING['lst_visualization'] = False to regenerate.")
else:
    # Wrap the src functions with @delayed for Dask parallel execution
    create_monthly_climatology_map_delayed = delayed(create_monthly_climatology_map)
    create_yearly_lst_map_delayed = delayed(create_yearly_lst_map)

    # Main execution using Dask
    print("\nðŸš€ Starting Dask-based visualization processing")
    print("="*70)

    # Create visualization tasks
    viz_tasks = []
    lst_time_options = ['day', 'night'] if LST_CONFIG['lst_time'] == 'both' else [LST_CONFIG['lst_time']]

    for city_name, aoi_info in aoi_data.items():
        city_dir = aoi_info['city_dir']

        for lst_time in lst_time_options:
            # Get visualization config for this time
            vis_config = VISUALIZATION_CONFIG[lst_time]

            # Create monthly climatology task
            monthly_task = create_monthly_climatology_map_delayed(
                city_name, city_dir, lst_time,
                dict(LST_CONFIG), dict(vis_config),
                dict(TEMPORAL_CONFIG), dict(PROJECT_CONFIG)
            )
            viz_tasks.append(monthly_task)

            # Create yearly map task
            yearly_task = create_yearly_lst_map_delayed(
                city_name, city_dir, lst_time,
                dict(LST_CONFIG), dict(vis_config),
                dict(TEMPORAL_CONFIG), dict(PROJECT_CONFIG)
            )
            viz_tasks.append(yearly_task)

    print(f"Created {len(viz_tasks)} visualization tasks")
    print(f"Cities: {', '.join(aoi_data.keys())}")
    print(f"Time periods: {', '.join(lst_time_options)}")

    # Submit tasks to Dask
    print(f"\n Submitting visualization tasks to Dask cluster...")
    print(f"Workers: {len(client.scheduler_info()['workers'])}")
    print(f"Dashboard: {client.dashboard_link}")

    futures = client.compute(viz_tasks)

    # Track progress
    results = []
    with tqdm(total=len(viz_tasks), desc="Creating visualizations") as pbar:
        for future, result in as_completed(futures, with_results=True):
            results.append(result)

            if result['status'] == 'success':
                viz_type = result['type']
                pbar.set_postfix_str(f" {result['city']} - {result['lst_time']} {viz_type}")
            elif result['status'] == 'error':
                pbar.set_postfix_str(f"Error: {result['city']} - {result['lst_time']} {result['type']}")
            elif result['status'] == 'no_data':
                pbar.set_postfix_str(f" No data: {result['city']}")

            pbar.update(1)

    # Summary
    print("\n" + "="*70)
    print(" VISUALIZATION SUMMARY")
    print("="*70)

    successful = [r for r in results if r['status'] == 'success']
    errors = [r for r in results if r['status'] == 'error']
    no_data = [r for r in results if r['status'] == 'no_data']

    print(f"Total tasks: {len(results)}")
    print(f"Successful: {len(successful)}")
    print(f"No data: {len(no_data)}")
    print(f"Errors: {len(errors)}")

    # Group by city and type
    if successful:
        print("\n SUCCESSFULLY CREATED:")
        for city in sorted(aoi_data.keys()):
            city_results = [r for r in successful if r['city'] == city]
            if city_results:
                print(f"\n{city}:")
                for result in city_results:
                    print(f"  - {result['lst_time'].title()}-time {result['type']} map")
                    print(f"    SVG: {os.path.basename(result['output_pdf'])}")

    if errors:
        print("\nERRORS:")
        for error in errors[:5]:
            print(f"\n{error['city']} - {error['lst_time']} {error['type']}:")
            print(f"  {error.get('message', 'Unknown error')[:200]}")

    print("\n Visualization processing complete!")
    print(f"Check figures in each city's LST_Temporal_Analysis/[Day|Night]/Figures directory")
    print("="*70)

In [ ]:
print("="*70)
print("QA MAP VISUALIZATION")
print("="*70)

import os
import glob
import re
import traceback
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.gridspec as gridspec
from src import data_io

# Import QA helper functions from src (previously defined inline)
from src.visualization import load_qa_geotiff, calculate_detailed_qa_stats

# --- Skip logic: check if all expected QA map outputs already exist ---
_skip_qa = config.SKIP_EXISTING.get('qa_maps', False)
_qa_all_exist = False

if _skip_qa:
    _qa_all_exist = True
    _lt_opts = ['day', 'night'] if LST_CONFIG['lst_time'] == 'both' else [LST_CONFIG['lst_time']]
    for _cn in aoi_data:
        for _lt in _lt_opts:
            _sub = 'Day' if _lt == 'day' else 'Night'
            _qa_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], f"City_{_cn}",
                                   'LST_Temporal_Analysis', _sub, 'Figures', 'QA_Maps')
            _expected = [
                os.path.join(_qa_dir, f"{_cn}_{_lt.title()}_Monthly_QA_Climatology.svg"),
                os.path.join(_qa_dir, f"{_cn}_{_lt.title()}_Monthly_QA_Climatology.pdf"),
                os.path.join(_qa_dir, f"{_cn}_{_lt.title()}_Yearly_QA.svg"),
                os.path.join(_qa_dir, f"{_cn}_{_lt.title()}_Yearly_QA.pdf"),
            ]
            if not all(os.path.exists(path) for path in _expected):
                _qa_all_exist = False
                break
        if not _qa_all_exist:
            break

if _skip_qa and _qa_all_exist:
    print("SKIPPED all QA map outputs already exist.")
    print("   Set config.SKIP_EXISTING['qa_maps'] = False to regenerate.")
else:
    # Define A5 size in inches for consistent figure size
    if 'A5_SIZE_INCHES' not in globals():
        A5_SIZE_INCHES = (8.27, 11.69)

    # Create QA colormap based on configuration
    qa_colors = []
    qa_labels = []
    qa_values = []

    # Build colors and labels from configuration
    if 'LST_CONFIG' in globals() and LST_CONFIG.get('qa_levels_to_use'):
        for tier in sorted(LST_CONFIG['qa_levels_to_use'], key=lambda t: LST_CONFIG['quality_hierarchy'][t]['priority']):
            tier_config = LST_CONFIG['quality_hierarchy'][tier]
            qa_colors.append(tier_config['color'])
            if 'description' in tier_config and '≤' in tier_config['description']:
                qa_labels.append(f"{tier.capitalize()} (≤{tier_config['description'].split('≤')[1].strip()})")
            else:
                qa_labels.append(tier.capitalize())
            qa_values.append(tier_config['priority'])

    if qa_colors and qa_values:
        qa_cmap = ListedColormap(qa_colors)
        qa_bounds = [qa_values[0] - 0.5] + [v + 0.5 for v in qa_values]
        qa_norm = BoundaryNorm(qa_bounds, len(qa_colors))
    else:
        print("QA configuration is missing or empty.")
        qa_cmap = None
        qa_bounds = None
        qa_norm = None

    # Process one city at a time for QA maps
    for city_name, aoi_info in aoi_data.items():
        print(f"\nProcessing QA visualizations for {city_name}")
        city_dir = aoi_info['city_dir']

        for lst_time in lst_time_options:
            print(f"\n   Processing {lst_time}-time QA visualizations")

            subdir = 'Day' if lst_time == 'day' else 'Night'
            geotiff_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], f"City_{city_name}", 'LST_Temporal_Analysis', subdir, 'geotiff')

            if not os.path.exists(geotiff_dir):
                print(f"  Geotiff directory not found: {geotiff_dir}")
                continue

            print(f"   Found geotiff directory: {geotiff_dir}")

            figures_dir = os.path.join(os.path.dirname(geotiff_dir), 'Figures')
            os.makedirs(figures_dir, exist_ok=True)

            qa_maps_dir = os.path.join(figures_dir, 'QA_Maps')
            os.makedirs(qa_maps_dir, exist_ok=True)

            # === MONTHLY QA CLIMATOLOGY MAP ===
            monthly_qa_pattern = os.path.join(geotiff_dir, f"{city_name}_QA_Source_{subdir.lower()}_MonthlyClimatology_*.tif")
            alt_monthly_pattern = os.path.join(geotiff_dir, f"{city_name}_QA_Source_{subdir.lower()}_month_*.tif")

            monthly_qa_files = sorted(glob.glob(monthly_qa_pattern))
            if not monthly_qa_files:
                monthly_qa_files = sorted(glob.glob(alt_monthly_pattern))
            if not monthly_qa_files:
                monthly_qa_pattern = os.path.join(geotiff_dir, f"{city_name}_month_*_QA_Source_{lst_time}_climatology.tif")
                monthly_qa_files = sorted(glob.glob(monthly_qa_pattern))

            print(f"  ðŸ” Searching for monthly QA files")

            if not monthly_qa_files:
                print(f"  No monthly QA climatology files found for {city_name} ({lst_time}-time)")
            else:
                print(f" Found {len(monthly_qa_files)} monthly QA climatology files")

                try:
                    print(f" Creating monthly QA climatology map...")

                    fig = plt.figure(figsize=A5_SIZE_INCHES, dpi=300)
                    gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.15, wspace=0.1)

                    month_names = ['January', 'February', 'March', 'April', 'May', 'June',
                                   'July', 'August', 'September', 'October', 'November', 'December']

                    for file_path in monthly_qa_files:
                        file_name = os.path.basename(file_path)
                        month_match = re.search(r'MonthlyClimatology_(\d+)', file_name) or \
                                      re.search(r'month_(\d+)', file_name) or \
                                      re.search(r'_(\d{2})\.tif', file_name)

                        if not month_match:
                            print(f" Warning: Could not extract month from filename: {file_name}")
                            continue

                        month_num = int(month_match.group(1))
                        row = (month_num - 1) // 3
                        col = (month_num - 1) % 3

                        ax = fig.add_subplot(gs[row, col])

                        qa_data, bounds, transform, crs = load_qa_geotiff(file_path)

                        if qa_data is not None:
                            im = ax.imshow(qa_data, cmap=qa_cmap, norm=qa_norm)
                            ax.set_title(month_names[month_num-1], fontsize=8)
                            ax.set_xticks([])
                            ax.set_yticks([])

                            # Pass LST_CONFIG to the imported function
                            qa_stats_str = calculate_detailed_qa_stats(qa_data, LST_CONFIG)

                            ax.text(0.98, 0.02, qa_stats_str, ha='right', va='bottom',
                                    transform=ax.transAxes, fontsize=7.5,
                                    bbox=dict(facecolor='white', alpha=0.7, pad=1, edgecolor='none'))
                        else:
                            ax.text(0.5, 0.5, 'No QA Data', ha='center', va='center', transform=ax.transAxes)
                            ax.set_title(month_names[month_num-1], fontsize=8)

                    cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
                    cbar = plt.colorbar(
                        plt.cm.ScalarMappable(cmap=qa_cmap, norm=qa_norm),
                        cax=cbar_ax,
                        orientation='horizontal',
                        ticks=qa_values
                    )
                    cbar.ax.set_xticklabels(qa_labels, fontsize=6)
                    cbar.set_label('QA Level', fontsize=8)

                    fig.suptitle(f"{city_name} - {lst_time.title()}-time Monthly QA Climatology", fontsize=10, y=0.95)
                    years_range = f"{TEMPORAL_CONFIG['years'][0]}-{TEMPORAL_CONFIG['years'][-1]}"
                    plt.figtext(0.5, 0.01, f"Data period: {years_range}", ha='center', fontsize=6)

                    output_stem = os.path.join(qa_maps_dir, f"{city_name}_{lst_time.title()}_Monthly_QA_Climatology")
                    output_paths = data_io.export_figure_set(
                        fig,
                        output_stem,
                        dpi=300,
                        include_svg=True,
                        include_pdf=True,
                    )
                    plt.close(fig)

                    print(f" Saved monthly QA climatology map to {output_paths['svg']}")
                    print(f"    and {output_paths['pdf']}")

                except Exception as e:
                    print(f"  Error creating monthly QA maps for {city_name} ({lst_time}-time): {e}")
                    traceback_str = traceback.format_exc()
                    print(f" Traceback: {traceback_str}")

            # === YEARLY QA MAP ===
            yearly_qa_pattern = os.path.join(geotiff_dir, f"{city_name}_QA_Source_{subdir.lower()}_Yearly_*.tif")
            alt_yearly_pattern = os.path.join(geotiff_dir, f"{city_name}_QA_Source_{subdir.lower()}_yearly_*.tif")

            yearly_qa_files = sorted(glob.glob(yearly_qa_pattern))
            if not yearly_qa_files:
                yearly_qa_files = sorted(glob.glob(alt_yearly_pattern))
            if not yearly_qa_files:
                yearly_qa_pattern = os.path.join(geotiff_dir, f"{city_name}_*_QA_Source_{lst_time}_yearly.tif")
                yearly_qa_files = sorted(glob.glob(yearly_qa_pattern))

            print(f"\n  Searching for yearly QA files")

            if not yearly_qa_files:
                print(f"  No yearly QA files found for {city_name} ({lst_time}-time)")
                continue

            print(f" Found {len(yearly_qa_files)} yearly QA files")

            try:
                n_years = len(yearly_qa_files)

                if n_years <= 12:
                    n_rows = 3
                    n_cols = 4
                elif n_years <= 16:
                    n_rows = 4
                    n_cols = 4
                elif n_years <= 20:
                    n_rows = 5
                    n_cols = 4
                else:
                    n_rows = 5
                    n_cols = 5

                fig = plt.figure(figsize=A5_SIZE_INCHES, dpi=300)
                gs = gridspec.GridSpec(n_rows, n_cols, figure=fig, hspace=0.15, wspace=0.1)

                for i, file_path in enumerate(yearly_qa_files):
                    file_name = os.path.basename(file_path)
                    year_match = re.search(r'Yearly_(\d{4})', file_name) or \
                                 re.search(r'yearly_(\d{4})', file_name) or \
                                 re.search(r'_(\d{4})_', file_name)

                    if not year_match:
                        print(f"   Warning: Could not extract year from filename: {file_name}")
                        continue

                    year_str = year_match.group(1)
                    row = i // n_cols
                    col = i % n_cols

                    if row >= n_rows:
                        print(f"   Warning: More years ({n_years}) than grid capacity ({n_rows*n_cols})")
                        print(f"      Skipping year {year_str} and subsequent years")
                        break

                    ax = fig.add_subplot(gs[row, col])

                    qa_data, bounds, transform, crs = load_qa_geotiff(file_path)

                    if qa_data is not None:
                        im = ax.imshow(qa_data, cmap=qa_cmap, norm=qa_norm)
                        ax.set_title(year_str, fontsize=8)
                        ax.set_xticks([])
                        ax.set_yticks([])

                        qa_stats_str = calculate_detailed_qa_stats(qa_data, LST_CONFIG)

                        ax.text(0.98, 0.02, qa_stats_str, ha='right', va='bottom',
                                transform=ax.transAxes, fontsize=7.5,
                                bbox=dict(facecolor='white', alpha=0.7, pad=1, edgecolor='none'))
                    else:
                        ax.text(0.5, 0.5, 'No QA Data', ha='center', va='center', transform=ax.transAxes)
                        ax.set_title(year_str, fontsize=8)

                cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
                cbar = plt.colorbar(
                    plt.cm.ScalarMappable(cmap=qa_cmap, norm=qa_norm),
                    cax=cbar_ax,
                    orientation='horizontal',
                    ticks=qa_values
                )
                cbar.ax.set_xticklabels(qa_labels, fontsize=6)
                cbar.set_label('QA Level', fontsize=8)

                fig.suptitle(f"{city_name} - {lst_time.title()}-time Yearly QA Maps", fontsize=10, y=0.95)

                output_stem = os.path.join(qa_maps_dir, f"{city_name}_{lst_time.title()}_Yearly_QA")
                output_paths = data_io.export_figure_set(
                    fig,
                    output_stem,
                    dpi=300,
                    include_svg=True,
                    include_pdf=True,
                )
                plt.close(fig)

                print(f" Saved yearly QA map to {output_paths['svg']}")
                print(f"    and {output_paths['pdf']}")

            except Exception as e:
                print(f"  Error creating yearly QA maps for {city_name} ({lst_time}-time): {e}")
                traceback_str = traceback.format_exc()
                print(f"  ðŸ“‹ Traceback: {traceback_str}")

    print("\n" + "="*70)
    print(" QA MAP VISUALIZATION COMPLETE")
    print("="*70)
    print("Check each city's QA maps in the Figures/QA_Maps directory")
    print("="*70)

## Block 6: Compute LST Statistics

Extracts comprehensive statistics from GeoTIFF files:

**Data Filtering:**
- Excludes LST < 10Â°C (likely water bodies, cloud artifacts, no-data)
- Retains only valid land surface temperatures

**Computed Metrics:**

For each city, month/year, and time period (day/night):

| Metric | Formula | Description |
|--------|---------|-------------|
| Mean | $\bar{x} = \frac{1}{n}\sum_{i=1}^{n}x_i$ | Average temperature |
| Std Dev | $\sigma = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2}$ | Temperature variability |
| Min/Max | - | Temperature extremes |
| Range | $R = x_{max} - x_{min}$ | Thermal amplitude |
| Median | $\tilde{x}$ | 50th percentile |

**Output Files:**
- Per-city CSV: `City_*/LST_Temporal_Analysis/[Day/Night]/Statistics/`
- Combined CSV: `Cross_City_Statistics/[monthly/yearly]_LST_statistics_[day/night].csv`

In [ ]:
print("="*70)
print("BLOCK 6: LST SUMMARY WITH YEARLY STATISTICS")
print("="*70)

# Start timing for enhanced statistics
final_stats_start_time = time.time()

# Define the main statistics directory - use a new "Stats" folder
main_stats_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], 'Stats')
os.makedirs(main_stats_dir, exist_ok=True)
print(f"\nMain statistics directory: {main_stats_dir}")

# --- Skip logic: check if key outputs already exist ---
_skip_stats = config.SKIP_EXISTING.get('lst_summary_stats', False)
_stats_outputs_exist = False

if _skip_stats:
    _req_files = [
        os.path.join(main_stats_dir, "All_Cities_LST_Statistics.csv"),
        os.path.join(main_stats_dir, "City_LST_Enhanced_Summary.csv"),
        os.path.join(main_stats_dir, "City_LST_Enhanced_Summary_Wide.csv"),
    ]
    _stats_outputs_exist = all(os.path.exists(f) for f in _req_files)

if _skip_stats and _stats_outputs_exist:
    print("  SKIPPED â€” all LST summary statistics outputs already exist.")
    print("   Set config.SKIP_EXISTING['lst_summary_stats'] = False to regenerate.")
    # Still load the files so downstream cells can use them
    all_stats_file = os.path.join(main_stats_dir, "All_Cities_LST_Statistics.csv")
    all_stats_df = pd.read_csv(all_stats_file)
    print(f"   Loaded existing statistics: {len(all_stats_df)} records")
else:
    # Create a better structured summary file with more statistics
    print("\nCreating enhanced summary report with yearly statistics and additional metrics")

    try:
        # Path for our target statistics file in the new location
        target_stats_file = os.path.join(main_stats_dir, "All_Cities_LST_Statistics.csv")
        
        # First, check possible locations for the statistics file
        possible_source_files = [
            # Check in LST_Temporal_Analysis_Dask directory
            os.path.join(PROJECT_CONFIG['lst_output_dir'], 'LST_Temporal_Analysis_Dask', 'Statistics', "All_Cities_LST_Statistics.csv"),
            # Check in each city's directory
            os.path.join(PROJECT_CONFIG['lst_output_dir'], 'LST_Temporal_Analysis_Dask', 'Statistics', "City_LST_Statistics.csv"),
        ]
        
        # Variable to store if we found and copied an existing file
        file_found = False
        
        # Check if any of these files exist and copy the first one found
        for source_file in possible_source_files:
            if os.path.exists(source_file):
                import shutil
                shutil.copy2(source_file, target_stats_file)
                print(f" Copied statistics file from {source_file} to: {target_stats_file}")
                file_found = True
                break
        
        # If we didn't find any existing files, collect and merge individual city statistics files
        if not file_found:
            print(" No existing statistics summary file found. Computing statistics from GeoTIFF files...")
            
            # Compute statistics directly from GeoTIFF files
            all_city_stats = []
            
            for city_name, aoi_info in aoi_data.items():
                city_dir = aoi_info['city_dir']
                print(f"\n Computing statistics for {city_name}...")
                
                # Process both day and night
                for lst_time in lst_time_options:
                    lst_subfolder = 'Day' if lst_time == 'day' else 'Night'
                    city_lst_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], f"City_{city_name}", 'LST_Temporal_Analysis', lst_subfolder)
                    geotiff_dir = os.path.join(city_lst_dir, 'geotiff')
                    
                    if not os.path.exists(geotiff_dir):
                        print(f"  Directory not found: {geotiff_dir}")
                        continue
                    
                    # Process yearly files
                    yearly_pattern = os.path.join(geotiff_dir, f"{city_name}_LST_{lst_time.title()}_Yearly_*.tif")
                    yearly_files = sorted(glob.glob(yearly_pattern))
                    
                    for file_path in yearly_files:
                        try:
                            year_match = re.search(r'Yearly_(\d{4})', file_path)
                            if not year_match:
                                continue
                            year = int(year_match.group(1))
                            
                            # Read GeoTIFF and compute statistics
                            with rasterio.open(file_path) as src:
                                data = src.read(1)
                                if src.nodata is not None:
                                    data = np.where(data == src.nodata, np.nan, data)
                                # Apply realistic temperature bounds for Nigerian cities
                                data = np.where((data < 10) | (data > 50), np.nan, data)
                                
                                valid_data = data[~np.isnan(data)]
                                if len(valid_data) > 0:
                                    stats_row = {
                                        'city': city_name,
                                        'day_night': lst_time,
                                        'time_period_type': 'yearly',
                                        'time_period': year,
                                        'is_climatology': False,
                                        'lst_min': float(np.min(valid_data)),
                                        'lst_max': float(np.max(valid_data)),
                                        'lst_mean': float(np.mean(valid_data)),
                                        'lst_median': float(np.median(valid_data)),
                                        'lst_std': float(np.std(valid_data)),
                                        'lst_range': float(np.max(valid_data) - np.min(valid_data)),
                                        'valid_pixel_count': len(valid_data)
                                    }
                                    all_city_stats.append(stats_row)
                                    print(f" {year} ({lst_time}): median={stats_row['lst_median']:.2f} °C")
                        except Exception as e:
                            print(f"  âœ— Error processing {file_path}: {e}")
                    
                    # Process monthly climatology files
                    monthly_pattern = os.path.join(geotiff_dir, f"{city_name}_LST_{lst_time.title()}_MonthlyClimatology_*.tif")
                    monthly_files = sorted(glob.glob(monthly_pattern))
                    
                    for file_path in monthly_files:
                        try:
                            month_match = re.search(r'MonthlyClimatology_(\d+)', file_path)
                            if not month_match:
                                continue
                            month = int(month_match.group(1))
                            
                            # Read GeoTIFF and compute statistics
                            with rasterio.open(file_path) as src:
                                data = src.read(1)
                                if src.nodata is not None:
                                    data = np.where(data == src.nodata, np.nan, data)
                                # Apply realistic temperature bounds for Nigerian cities
                                data = np.where((data < 10) | (data > 50), np.nan, data)
                                
                                valid_data = data[~np.isnan(data)]
                                if len(valid_data) > 0:
                                    stats_row = {
                                        'city': city_name,
                                        'day_night': lst_time,
                                        'time_period_type': 'monthly',
                                        'time_period': month,
                                        'is_climatology': True,
                                        'lst_min': float(np.min(valid_data)),
                                        'lst_max': float(np.max(valid_data)),
                                        'lst_mean': float(np.mean(valid_data)),
                                        'lst_median': float(np.median(valid_data)),
                                        'lst_std': float(np.std(valid_data)),
                                        'lst_range': float(np.max(valid_data) - np.min(valid_data)),
                                        'valid_pixel_count': len(valid_data)
                                    }
                                    all_city_stats.append(stats_row)
                        except Exception as e:
                            print(f" Error processing {file_path}: {e}")
                
            if all_city_stats:
                # Create DataFrame from computed statistics
                all_stats_df = pd.DataFrame(all_city_stats)
                
                # Save the combined file
                all_stats_df.to_csv(target_stats_file, index=False)
                print(f"\n Created statistics file with {len(all_stats_df)} records: {target_stats_file}")
                file_found = True
            else:
                print("\n No statistics could be computed from GeoTIFF files")
                
        # Use the new target file location
        all_stats_file = target_stats_file
        
        if os.path.exists(all_stats_file) and file_found:
            all_stats_df = pd.read_csv(all_stats_file)
            
            # Prepare data for summary table with enhanced statistics
            summary_rows = []
            
            # Process each city
            for city_name, aoi_info in aoi_data.items():
                # Filter for this city's data
                city_df = all_stats_df[all_stats_df['city'] == city_name]
                
                if city_df.empty:
                    print(f" No data found for {city_name} in the statistics file")
                    continue
                    
                # Process both day and night
                for lst_time in lst_time_options:
                    # Filter for day/night data
                    time_df = city_df[city_df['day_night'] == lst_time]
                    
                    if time_df.empty:
                        print(f" No {lst_time}-time data found for {city_name}")
                        continue
                    
                    time_str = lst_time.capitalize()
                    
                    # --- MONTHLY CLIMATOLOGY STATISTICS ---
                    monthly_df = time_df[(time_df['time_period_type'] == 'monthly') & (time_df['is_climatology'] == True)]
                    
                    if monthly_df.empty:
                        monthly_df = time_df[time_df['time_period_type'] == 'monthly']
                        print(f" No climatology data found for {city_name} {lst_time}, using regular monthly data")
                    
                    if not monthly_df.empty:
                        # Calculate overall statistics
                        mean_lst = monthly_df['lst_mean'].mean()
                        median_lst = monthly_df['lst_median'].mean()  # Average of the monthly medians
                        min_lst = monthly_df['lst_mean'].min()
                        max_lst = monthly_df['lst_mean'].max()
                        lst_range = max_lst - min_lst
                        overall_min = monthly_df['lst_min'].min()  # Absolute minimum LST
                        overall_max = monthly_df['lst_max'].max()  # Absolute maximum LST
                        
                        # Calculate additional statistics
                        lst_std = monthly_df['lst_mean'].std()  # Standard deviation of monthly means
                        warmest_quarter = monthly_df.nlargest(3, 'lst_mean')['lst_mean'].mean()  # Mean of 3 warmest months
                        coolest_quarter = monthly_df.nsmallest(3, 'lst_mean')['lst_mean'].mean()  # Mean of 3 coolest months
                        seasonal_amplitude = warmest_quarter - coolest_quarter
                        
                        # Find hottest month (using mean LST)
                        hottest_idx = monthly_df['lst_mean'].idxmax()
                        hottest_month_num = monthly_df.loc[hottest_idx, 'time_period']
                        hottest_month_lst = monthly_df.loc[hottest_idx, 'lst_mean']
                        
                        # Month with highest maximum temperature
                        max_temp_idx = monthly_df['lst_max'].idxmax()
                        max_temp_month_num = monthly_df.loc[max_temp_idx, 'time_period']
                        max_temp_value = monthly_df.loc[max_temp_idx, 'lst_max']
                        
                        # Convert month numbers to names
                        month_names_dict = {
                            1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
                            7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
                        }
                        hottest_month = month_names_dict.get(hottest_month_num, str(hottest_month_num))
                        max_temp_month = month_names_dict.get(max_temp_month_num, str(max_temp_month_num))
                        
                        # Find coolest month (using mean LST)
                        coolest_idx = monthly_df['lst_mean'].idxmin()
                        coolest_month_num = monthly_df.loc[coolest_idx, 'time_period']
                        coolest_month_lst = monthly_df.loc[coolest_idx, 'lst_mean']
                        coolest_month = month_names_dict.get(coolest_month_num, str(coolest_month_num))
                        
                        # --- YEARLY STATISTICS ---
                        yearly_df = time_df[time_df['time_period_type'] == 'yearly']
                        
                        # Initialize yearly statistics
                        hottest_year = "N/A"
                        hottest_year_lst = 0
                        coolest_year = "N/A"
                        coolest_year_lst = 0
                        warming_rate = "N/A"
                        trend_significance = "N/A"
                        
                        if not yearly_df.empty:
                            # Find hottest year
                            hottest_year_idx = yearly_df['lst_mean'].idxmax()
                            hottest_year = yearly_df.loc[hottest_year_idx, 'time_period']
                            hottest_year_lst = yearly_df.loc[hottest_year_idx, 'lst_mean']
                            
                            # Find coolest year
                            coolest_year_idx = yearly_df['lst_mean'].idxmin()
                            coolest_year = yearly_df.loc[coolest_year_idx, 'time_period']
                            coolest_year_lst = yearly_df.loc[coolest_year_idx, 'lst_mean']
                            
                            # Calculate warming rate if we have enough years
                            if len(yearly_df) > 3:
                                # Sort by year
                                yearly_df = yearly_df.sort_values('time_period')
                                
                                # Simple linear regression to calculate warming rate
                                years = yearly_df['time_period'].values
                                temps = yearly_df['lst_mean'].values
                                
                                if len(years) == len(temps) and len(years) > 1:
                                    try:
                                        from scipy import stats
                                        # Linear regression with significance test
                                        slope, intercept, r_value, p_value, std_err = stats.linregress(years, temps)
                                        
                                        # Format warming rate with RÂ² and significance indication
                                        r_squared = r_value**2
                                        
                                        # Check significance at 0.05 threshold
                                        if p_value < 0.05:
                                            significance = "significant"
                                            sig_marker = "*"
                                        else:
                                            significance = "not significant"
                                            sig_marker = ""
                                            
                                        warming_rate = f"{slope:.3f} °C/year (RÂ²={r_squared:.2f}{sig_marker})"
                                        trend_significance = f"p={p_value:.3f} ({significance} at p=0.05 level)"
                                        
                                        # Also try Mann-Kendall test if available
                                        try:
                                            import pymannkendall as mk
                                            result = mk.original_test(temps)
                                            if result.p < 0.05:
                                                mk_sig = "significant"
                                                mk_marker = "*"
                                            else:
                                                mk_sig = "not significant"
                                                mk_marker = ""
                                                
                                            # Add Mann-Kendall test result
                                            trend_significance += f"\nMann-Kendall: {result.trend} trend{mk_marker} (p={result.p:.3f})"
                                        except:
                                            # Mann-Kendall test not available
                                            pass
                                            
                                    except:
                                        # If scipy is not available, use numpy for a simple approach
                                        years_mean = years.mean()
                                        temps_mean = temps.mean()
                                        numerator = sum((years - years_mean) * (temps - temps_mean))
                                        denominator = sum((years - years_mean) ** 2)
                                        slope = numerator / denominator if denominator != 0 else 0
                                        warming_rate = f"{slope:.3f} °C/year"
                        
                        # Create row with enhanced statistics
                        row = {
                            'City': city_name,
                            'Time': time_str,
                            'Mean Annual LST (°C)': round(mean_lst, 1),
                            'Median LST (°C)': round(median_lst, 1),
                            'Overall Min-Max (°C)': f"{round(overall_min, 1)} - {round(overall_max, 1)}",
                            'Monthly Range (°C)': f"{round(min_lst, 1)} - {round(max_lst, 1)}",
                            'Monthly Amplitude (°C)': round(lst_range, 1),
                            'Seasonal Amplitude (°C)': round(seasonal_amplitude, 1),
                            'Std Dev ( °C)': round(lst_std, 1),
                            'Hottest Month': f"{hottest_month} ({round(hottest_month_lst, 1)}°C)",
                            'Coolest Month': f"{coolest_month} ({round(coolest_month_lst, 1)}°C)",
                            'Max Temp Month': f"{max_temp_month} ({round(max_temp_value, 1)}°C)",
                            'Hottest Year': f"{hottest_year} ({round(hottest_year_lst, 1)}°C)" if hottest_year != "N/A" else "N/A",
                            'Coolest Year': f"{coolest_year} ({round(coolest_year_lst, 1)}°C)" if coolest_year != "N/A" else "N/A",
                            'Warming Rate': warming_rate,
                            'Trend Significance': trend_significance
                        }
                        
                        summary_rows.append(row)
                        
                        # Create detailed text summary for this city and time
                        summary_text = f"# {city_name} {time_str}-time LST Summary Statistics\n\n"
                        
                        try:
                            years_range = f"{TEMPORAL_CONFIG['years'][0]}-{TEMPORAL_CONFIG['years'][-1]}"
                        except:
                            years_range = "Study Period"
                        
                        summary_text += f"## Monthly Climatology ({years_range})\n"
                        summary_text += f"- Mean annual LST: {mean_lst:.1f} °C\n"
                        summary_text += f"- Median annual LST: {median_lst:.1f} °C\n"
                        summary_text += f"- Overall temperature range: {overall_min:.1f} °C - {overall_max:.1f} °C\n"
                        summary_text += f"- Monthly mean range: {min_lst:.1f} °C - {max_lst:.1f} °C\n"
                        summary_text += f"- Monthly amplitude: {lst_range:.1f} °C\n"
                        summary_text += f"- Seasonal amplitude: {seasonal_amplitude:.1f} °C\n"
                        summary_text += f"- Standard deviation: {lst_std:.1f} °C\n"
                        summary_text += f"- Hottest month: {hottest_month} ({hottest_month_lst:.1f} °C)\n"
                        summary_text += f"- Coolest month: {coolest_month} ({coolest_month_lst:.1f} °C)\n"
                        summary_text += f"- Month with highest max temperature: {max_temp_month} ({max_temp_value:.1f} °C)\n"
                        
                        if hottest_year != "N/A":
                            summary_text += f"\n Yearly Statistics\n"
                            summary_text += f"- Hottest year: {hottest_year} ({hottest_year_lst:.1f} °C)\n"
                            summary_text += f"- Coolest year: {coolest_year} ({coolest_year_lst:.1f} °C)\n"
                            summary_text += f"- Warming trend: {warming_rate}\n"
                            summary_text += f"- Trend significance: {trend_significance}\n"
                            summary_text += f"  * p-value < 0.05 indicates a statistically significant trend at 95% confidence level\n"
                        
                        # Save city-specific summary
                        city_dir = aoi_info['city_dir']
                        
                        # Use new directory structure with "Stats" as the base folder
                        city_stats_dir = os.path.join(main_stats_dir, city_name)
                        os.makedirs(city_stats_dir, exist_ok=True)
                        
                        # Create subfolders for day and night statistics
                        city_time_stats_dir = os.path.join(city_stats_dir, time_str)
                        os.makedirs(city_time_stats_dir, exist_ok=True)
                        
                        # Save enhanced summary text file - FIX: Add encoding parameter
                        summary_file = os.path.join(city_time_stats_dir, f"{city_name}_{time_str}_Enhanced_Summary.txt")
                        with open(summary_file, 'w', encoding='utf-8') as f:
                            f.write(summary_text)
                        print(f"   Saved enhanced {time_str} summary for {city_name} to {summary_file}")
            
            # Create enhanced summary dataframe
            if summary_rows:
                summary_df = pd.DataFrame(summary_rows)
                
                # Save long-format CSV (one row per city-time)
                summary_csv_path = os.path.join(main_stats_dir, "City_LST_Enhanced_Summary.csv")
                summary_df.to_csv(summary_csv_path, index=False)
                print(f" Saved long-format CSV to {summary_csv_path}")
                
                # -----------------------------------------------------------
                # Build WIDE table: one row per city, Day|Night side by side
                # -----------------------------------------------------------
                stat_cols = [c for c in summary_df.columns if c not in ('City', 'Time')]
                
                day_df = summary_df[summary_df['Time'] == 'Day'].set_index('City')[stat_cols]
                night_df = summary_df[summary_df['Time'] == 'Night'].set_index('City')[stat_cols]
                
                # Prefix columns with Day_ / Night_
                day_df = day_df.rename(columns={c: f"Day {c}" for c in stat_cols})
                night_df = night_df.rename(columns={c: f"Night {c}" for c in stat_cols})
                
                # Merge on City index
                wide_df = day_df.join(night_df, how='outer').reset_index()
                wide_df = wide_df.rename(columns={'index': 'City'})
                
                # Interleave Day/Night columns for readability
                ordered_cols = ['City']
                for c in stat_cols:
                    if f"Day {c}" in wide_df.columns:
                        ordered_cols.append(f"Day {c}")
                    if f"Night {c}" in wide_df.columns:
                        ordered_cols.append(f"Night {c}")
                wide_df = wide_df[[c for c in ordered_cols if c in wide_df.columns]]
                
                # Save wide CSV
                wide_csv_path = os.path.join(main_stats_dir, "City_LST_Enhanced_Summary_Wide.csv")
                wide_df.to_csv(wide_csv_path, index=False)
                print(f" Saved wide-format CSV to {wide_csv_path}")
                
                # Display the wide summary table
                print("\nENHANCED CITY LST SUMMARY (Day | Night side-by-side)")
                for _, row in wide_df.iterrows():
                    city = row['City']
                    print(f"\nðŸ“ {city}:")
                    for c in stat_cols:
                        day_val = row.get(f"Day {c}", "N/A")
                        night_val = row.get(f"Night {c}", "N/A")
                        print(f"  {c:30s}  Day: {day_val}  |  Night: {night_val}")
                
                # Styled HTML with Day/Night side-by-side
                # Identify numeric columns for gradient
                numeric_gradient_cols = [c for c in wide_df.columns 
                                         if any(kw in c for kw in ['Mean Annual', 'Monthly Amplitude', 'Seasonal Amplitude'])]
                
                styled_wide = wide_df.style
                if numeric_gradient_cols:
                    styled_wide = styled_wide.background_gradient(
                        subset=numeric_gradient_cols, cmap='RdYlBu_r'
                    )
                
                def highlight_significant_trends(val):
                    if isinstance(val, str) and '*' in val:
                        return 'color: #008800; font-weight: bold'
                    return ''
                
                warming_cols = [c for c in wide_df.columns if 'Warming Rate' in c]
                if warming_cols:
                    styled_wide = styled_wide.applymap(highlight_significant_trends, subset=warming_cols)
                
                html_path = os.path.join(main_stats_dir, "City_LST_Enhanced_Summary.html")
                styled_wide.to_html(html_path)
                print(f" Saved HTML summary (wide) to {html_path}")
        else:
            print(f"Could not find or create statistics file: {all_stats_file}")
            print(f"Make sure you have run Block 3 to generate the statistics files first.")
            
    except Exception as e:
        print(f"Error creating enhanced summary with yearly statistics: {e}")
        traceback_str = traceback.format_exc()
        print(f" Traceback: {traceback_str}")

# Calculate execution time
final_stats_end_time = time.time()
final_stats_duration = final_stats_end_time - final_stats_start_time
print("\n" + "="*70)
print(f"FINAL STATISTICS ENHANCEMENTS COMPLETED IN {final_stats_duration:.2f} SECONDS ({final_stats_duration/60:.2f} MINUTES)")
print("="*70)
print("Enhanced LST summary statistics with yearly data have been generated")
print("="*70)

## Block 7: Trend Analysis & Visualization

Creates dual-panel diagnostic plots for each city:

**Left Panel: Annual Trend**
- Time series of yearly mean LST (2006-2025)
- Linear regression trend line: $y = \beta_0 + \beta_1 \cdot t$
- 95% confidence interval (shaded region)
- Warming rate: $\Delta T = \beta_1 \times 10$  °C/decade
- Statistical significance: $p$-value and $R^2$
- Mann-Kendall test: $Z$-statistic and trend direction

**Right Panel: Monthly Climatology**
- Seasonal cycle (12 months)
- Shaded envelope: Min-Max temperature range
- Annotated peaks: Hottest and coolest months
- Summary statistics in text box


## Monthly Amplitude and Standard Deviation of LST

In addition to annual trend analysis, this notebook computes two key monthly metrics for land surface temperature (LST):

- **Monthly Amplitude:** The difference between the maximum and minimum mean LST for each month, capturing the range of surface warming across the annual cycle. This metric highlights the degree of seasonal variability and identifies months with the most pronounced temperature swings.
- **Monthly Standard Deviation:** The standard deviation of mean LST for each month, quantifying intra-month variability and the stability of surface temperatures. Higher values indicate greater fluctuation within a given month, which may reflect transient heat events or local climate anomalies.

These metrics provide a nuanced view of urban heat dynamics, revealing both the magnitude and variability of surface warming at the monthly scale. They are essential for understanding how cities experience heat across different times of year, and for identifying periods of heightened vulnerability to extreme temperatures.

**Interpretation:**
- Cities with high monthly amplitude may face sharp transitions between cool and hot periods, requiring adaptive planning for both extremes.
- Cities with high monthly standard deviation may experience unpredictable heat events, complicating risk management and public health responses.

The results are visualized and summarized below, supporting targeted interventions and climate resilience strategies.



**Color Scheme:**
- Day LST: Orange (#FF7F0E)
- Night LST: Black (#000000)
- Confidence bands: Semi-transparent fill

**Significance Markers:**
- (*) indicates $p < 0.05$
- Mann-Kendall trend: â†‘ (increasing), â†“ (decreasing), â†’ (no trend)

In [ ]:
print("="*70)
print("BLOCK 7: LST TREND ANALYSIS - IMPROVED VISUALIZATION")
print("="*70)

# Define the figure size for 1x2 plots (side by side layout with shared y-axis)
FIGURE_SIZE = (18, 9)  # Larger figure for better readability
DPI = 300

# Define colors using proper matplotlib RGBA format
DAY_COLOR = '#FF7F0E'  # Orange for day
DAY_FILL = (1.0, 0.5, 0.055, 0.3)  # Lighter orange with alpha=0.3
NIGHT_COLOR = '#000000'  # Black for night line and markers
NIGHT_FILL = (0.5, 0.5, 0.5, 0.3)  # Grey with alpha=0.3 for night shaded area

# Month abbreviations
MONTH_ABBRS = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

# Import helper functions from src (previously defined inline)
from src.visualization import find_safe_text_position, place_trend_box_safely, place_monthly_stats_box_safely

# --- Skip logic: check if all expected trend plot outputs already exist ---
_skip_trends = config.SKIP_EXISTING.get('trend_plots', False)
_trends_all_exist = False

if _skip_trends:
    _trends_all_exist = True
    _legend_svg = os.path.join(PROJECT_CONFIG['lst_output_dir'], 'Stats', 'LST_Trend_Plot_Legend.svg')
    _legend_pdf = os.path.join(PROJECT_CONFIG['lst_output_dir'], 'Stats', 'LST_Trend_Plot_Legend.pdf')
    if not (os.path.exists(_legend_svg) and os.path.exists(_legend_pdf)):
        _trends_all_exist = False
    else:
        for _cn in aoi_data:
            _fdir = os.path.join(PROJECT_CONFIG['lst_output_dir'], f'City_{_cn}',
                                 'LST_Temporal_Analysis', 'Figures')
            _exp_svg = os.path.join(_fdir, f"{_cn}_LST_Trends_Combined2.svg")
            _exp_pdf = os.path.join(_fdir, f"{_cn}_LST_Trends_Combined2.pdf")
            if not (os.path.exists(_exp_svg) and os.path.exists(_exp_pdf)):
                _trends_all_exist = False
                break

if _skip_trends and _trends_all_exist:
    print("SKIPPED - all LST trend plot outputs already exist.")
    print("Set config.SKIP_EXISTING['trend_plots'] = False to regenerate.")
else:
    try:
        # Set non-interactive matplotlib backend to avoid parallel execution errors
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.ticker as ticker  # For integer years formatting

        print("\nGenerating improved LST trend line plots for each city...")

        # Load the statistics file from Stats folder
        stats_file = os.path.join(PROJECT_CONFIG['lst_output_dir'], 'Stats', 'All_Cities_LST_Statistics.csv')

        if os.path.exists(stats_file):
            all_stats_df = pd.read_csv(stats_file)
            print(f"Loaded statistics from: {stats_file}")
        else:
            # Try alternative locations
            alt_stats_files = [
                os.path.join(PROJECT_CONFIG['lst_output_dir'], 'Stats', 'City_LST_Enhanced_Summary.csv'),
                os.path.join(PROJECT_CONFIG['lst_output_dir'], 'LST_Temporal_Analysis_Dask', 'Statistics', 'All_Cities_LST_Statistics.csv')
            ]

            stats_found = False
            for file in alt_stats_files:
                if os.path.exists(file):
                    all_stats_df = pd.read_csv(file)
                    print(f"Loaded statistics from alternative location: {file}")
                    stats_found = True
                    break

            if not stats_found:
                raise FileNotFoundError("Could not find statistics file in any known location.")

        # Determine which LST time periods to process
        lst_time_options = ['day', 'night'] if LST_CONFIG['lst_time'] == 'both' else [LST_CONFIG['lst_time']]

        # Process each city
        for city_name, aoi_info in aoi_data.items():
            print(f"\nProcessing trend plots for {city_name}...")

            # Filter for this city's data
            city_df = all_stats_df[all_stats_df['city'] == city_name]

            if city_df.empty:
                print(f"No data found for {city_name} in the statistics file")
                continue

            # Create directory for trend plots in the city's LST_Temporal_Analysis folder
            city_temporal_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], f'City_{city_name}', 'LST_Temporal_Analysis')
            os.makedirs(city_temporal_dir, exist_ok=True)

            # Create the figure with 1x2 subplots sharing y-axis
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIGURE_SIZE, sharey=True)

            # Reduce space between subplots
            plt.subplots_adjust(wspace=0.05)

            # Get study period for title
            try:
                years_range = f"({TEMPORAL_CONFIG['years'][0]}-{TEMPORAL_CONFIG['years'][-1]})"
            except Exception:
                years_range = ""

            # Add overall title with increased font size
            fig.suptitle(f"{city_name} Land Surface Temperature {years_range}",
                        fontweight='bold', fontsize=27, y=0.95)

            # Store data for both plots to determine y-axis limits and safe positioning
            all_temps = []
            day_data = {}
            night_data = {}

            # LEFT PLOT: Yearly Trends (both day and night)
            ax1.set_title("Yearly LST Trends", fontsize=21, pad=20)

            # Process yearly data for both day and night
            for lst_time in lst_time_options:
                # Filter for day/night data
                time_df = city_df[city_df['day_night'] == lst_time]

                if time_df.empty:
                    print(f"No {lst_time}-time data found for {city_name}")
                    continue

                # Get yearly data
                yearly_df = time_df[time_df['time_period_type'] == 'yearly']

                if yearly_df.empty:
                    print(f"No yearly data found for {city_name} {lst_time}")
                    continue

                # Sort yearly data chronologically
                yearly_df = yearly_df.sort_values('time_period')

                # Convert years to integers and get median LST values
                years = yearly_df['time_period'].values.astype(int)
                temps = yearly_df['lst_median'].values

                # Get min-max values for shaded area
                min_temps = yearly_df['lst_min'].values
                max_temps = yearly_df['lst_max'].values

                # Store all temperatures for y-axis limits
                all_temps.extend(min_temps)
                all_temps.extend(max_temps)

                # Get color and fill based on day/night
                if lst_time == 'day':
                    color = DAY_COLOR
                    fill_color = DAY_FILL
                    linestyle = ':'  # Dotted line for day trend
                    day_data = {
                        'years': years, 'temps': temps, 'min_temps': min_temps, 'max_temps': max_temps
                    }
                else:
                    color = NIGHT_COLOR
                    fill_color = NIGHT_FILL
                    linestyle = ':'  # Dotted line for night trend
                    night_data = {
                        'years': years, 'temps': temps, 'min_temps': min_temps, 'max_temps': max_temps
                    }

                time_label = lst_time.capitalize()

                # Plot shaded min-max range
                ax1.fill_between(years, min_temps, max_temps, color=fill_color, zorder=1)

                # Plot median line and points
                ax1.plot(years, temps, color=color, alpha=0.8, linestyle='-', linewidth=3.75, zorder=2)
                ax1.scatter(years, temps, color=color, s=60, zorder=3)

                # Calculate and plot trendline
                try:
                    from scipy import stats
                    slope, intercept, r_value, p_value, std_err = stats.linregress(years, temps)
                    r_squared = r_value**2

                    # Generate trend line
                    trend_line = intercept + slope * years

                    # Plot trendline with appropriate style
                    ax1.plot(years, trend_line, color=color, linestyle=linestyle, linewidth=3, zorder=1)

                    # Calculate total warming/cooling over the period
                    total_change = slope * (years[-1] - years[0])
                    warming_per_decade = slope * 10

                    # Create trend text box with increased font size
                    equation_text = f"{time_label}: y = {slope:.4f}x + {intercept:.2f}"
                    stats_text = f"R^2 = {r_squared:.3f}, p<{p_value:.2f}*" if p_value < 0.05 else f"R^2 = {r_squared:.3f}, p={p_value:.3f}"
                    warming_text = f"Warming: {warming_per_decade:.2f}C/decade ({total_change:.2f}C total)"

                    full_text = f"{equation_text}\n{stats_text}\n{warming_text}"

                    # Store trend info for safe positioning later
                    if lst_time == 'day':
                        day_data['trend_text'] = full_text
                        day_data['color'] = color
                    else:
                        night_data['trend_text'] = full_text
                        night_data['color'] = color

                except Exception as e:
                    print(f"Error calculating trend for {city_name} {lst_time}: {e}")

            # Place trend boxes safely with night always at bottom left
            box_positions = place_trend_box_safely(ax1, day_data if day_data else None, night_data if night_data else None)

            # Add the text boxes with fixed positions
            for lst_time in ['day', 'night']:
                data = day_data if lst_time == 'day' else night_data
                if data and 'trend_text' in data and lst_time in box_positions:
                    x_pos, y_pos = box_positions[lst_time]

                    # Set vertical alignment based on position (top for day, bottom for night)
                    v_align = 'top' if lst_time == 'day' else 'bottom'

                    ax1.text(x_pos, y_pos, data['trend_text'], transform=ax1.transAxes,
                            fontsize=15, verticalalignment=v_align,
                            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.6, edgecolor=data['color'], linewidth=2),
                            color=data['color'], fontweight='bold')

            # Add annotations for min/max values with safe positioning
            for lst_time in ['day', 'night']:
                data = day_data if lst_time == 'day' else night_data
                if not data:
                    continue

                years = data['years']
                temps = data['temps']
                color = DAY_COLOR if lst_time == 'day' else NIGHT_COLOR

                # Find hottest and coolest year values from yearly median LST
                max_year_idx = np.argmax(temps)
                max_year = years[max_year_idx]
                max_temp = temps[max_year_idx]
                min_year_idx = np.argmin(temps)
                min_year = years[min_year_idx]
                min_temp = temps[min_year_idx]

                # Calculate safe positions for annotations
                y_range = ax1.get_ylim()[1] - ax1.get_ylim()[0]
                offset = y_range * 0.03  # 3% of y-range

                # Annotate hottest year value with safe positioning
                safe_y_max = find_safe_text_position(ax1, max_year, max_temp, offset)
                ax1.annotate(f"{max_year}: {max_temp:.1f}C",
                           xy=(max_year, max_temp),
                           xytext=(max_year, safe_y_max),
                           ha='center', va='top',
                           fontsize=12, fontweight='bold', color=color,
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9),
                           arrowprops=dict(arrowstyle='->', color=color, lw=1))

                # Annotate minimum year value with safe positioning
                safe_y_min = find_safe_text_position(ax1, min_year, min_temp, -offset)
                ax1.annotate(f"{min_year}: {min_temp:.1f}C",
                           xy=(min_year, min_temp),
                           xytext=(min_year, safe_y_min),
                           ha='center', va='top',
                           fontsize=12, fontweight='bold', color=color,
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9),
                           arrowprops=dict(arrowstyle='->', color=color, lw=1))

            # Format left plot - Ensure years are displayed as integers
            ax1.set_xlabel("Year", fontsize=18)
            ax1.set_ylabel("LST (C)", fontsize=18)
            ax1.grid(True, linestyle='--', alpha=0.6)
            ax1.tick_params(axis='both', which='major', labelsize=15)

            # Set integer years on x-axis
            ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
            ax1.xaxis.set_major_formatter(ticker.FormatStrFormatter('%d'))

            # RIGHT PLOT: Monthly Climatology (both day and night)
            ax2.set_title("Monthly LST Climatology", fontsize=21, pad=20)

            # Reset data storage for monthly plot
            day_data = {}
            night_data = {}

            # Process monthly climatology data for both day and night
            for lst_time in lst_time_options:
                # Filter for day/night data
                time_df = city_df[city_df['day_night'] == lst_time]

                if time_df.empty:
                    continue

                # Get monthly climatology data
                monthly_df = time_df[(time_df['time_period_type'] == 'monthly') & (time_df['is_climatology'] == True)]

                if monthly_df.empty:
                    monthly_df = time_df[time_df['time_period_type'] == 'monthly']
                    print(f"No climatology data found for {city_name} {lst_time}, using regular monthly data")

                if monthly_df.empty:
                    continue

                # Sort by month number
                monthly_df = monthly_df.sort_values('time_period')
                months = monthly_df['time_period'].values.astype(int)
                temps = monthly_df['lst_median'].values

                # Get min-max values for shaded area
                min_temps = monthly_df['lst_min'].values
                max_temps = monthly_df['lst_max'].values

                # Store all temperatures for consistent y-axis
                all_temps.extend(min_temps)
                all_temps.extend(max_temps)

                # Get color based on day/night
                if lst_time == 'day':
                    color = DAY_COLOR
                    fill_color = DAY_FILL
                    day_data = {'months': months, 'temps': temps, 'min_temps': min_temps, 'max_temps': max_temps}
                else:
                    color = NIGHT_COLOR
                    fill_color = NIGHT_FILL
                    night_data = {'months': months, 'temps': temps, 'min_temps': min_temps, 'max_temps': max_temps}

                # Plot shaded min-max range
                ax2.fill_between(months, min_temps, max_temps, color=fill_color, zorder=1)

                # Plot median line and points
                ax2.plot(months, temps, color=color, alpha=0.8, linestyle='-', linewidth=3.75, zorder=2)
                ax2.scatter(months, temps, color=color, s=60, zorder=3)

                # Find and annotate extreme months with safe positioning
                min_idx = np.argmin(temps)
                max_idx = np.argmax(temps)

                min_month = MONTH_ABBRS[months[min_idx]]
                max_month = MONTH_ABBRS[months[max_idx]]

                # Calculate safe positions
                y_range = np.max(all_temps) - np.min(all_temps) if all_temps else 10
                offset = y_range * 0.05  # 5% of y-range

                # Annotate maximum month with safe positioning
                safe_y_max = find_safe_text_position(ax2, months[max_idx], temps[max_idx], offset)
                ax2.annotate(f"{max_month}: {temps[max_idx]:.1f}C",
                           xy=(months[max_idx], temps[max_idx]),
                           xytext=(months[max_idx], safe_y_max),
                           ha='center', va='top',
                           fontsize=15, fontweight='bold', color=color,
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9),
                           arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

                # Annotate minimum month with safe positioning
                safe_y_min = find_safe_text_position(ax2, months[min_idx], temps[min_idx], -offset)
                ax2.annotate(f"{min_month}: {temps[min_idx]:.1f}C",
                           xy=(months[min_idx], temps[min_idx]),
                           xytext=(months[min_idx], safe_y_min),
                           ha='center', va='top',
                           fontsize=15, fontweight='bold', color=color,
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9),
                           arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

                # Calculate additional monthly statistics
                monthly_amplitude = np.max(temps) - np.min(temps)
                monthly_std = np.std(temps)

                # Create simplified statistics text box with only MA and Std Dev
                monthly_stats_text = f"Monthly amplitude: {monthly_amplitude:.1f}C\nStd Dev: {monthly_std:.1f}C"

                # Store monthly stats info for positioning later
                if lst_time == 'day':
                    day_data['monthly_stats_text'] = monthly_stats_text
                    day_data['color'] = color
                else:
                    night_data['monthly_stats_text'] = monthly_stats_text
                    night_data['color'] = color

            # Place monthly statistics boxes safely with night always at bottom right
            monthly_box_positions = place_monthly_stats_box_safely(ax2, day_data if day_data else None, night_data if night_data else None)

            # Add monthly statistics boxes to the plot with fixed positions
            for lst_time in ['day', 'night']:
                data = day_data if lst_time == 'day' else night_data
                if data and 'monthly_stats_text' in data and lst_time in monthly_box_positions:
                    x_pos, y_pos = monthly_box_positions[lst_time]

                    # Set vertical alignment based on position (top for day, bottom for night)
                    v_align = 'top' if lst_time == 'day' else 'bottom'

                    ax2.text(x_pos, y_pos, data['monthly_stats_text'], transform=ax2.transAxes,
                            fontsize=12, verticalalignment=v_align, horizontalalignment='right',
                            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.6, edgecolor=data['color'], linewidth=1),
                            color=data['color'], fontweight='bold')

            # Format right plot (no y-label since it's shared)
            ax2.set_xlabel("Month", fontsize=18)
            ax2.set_xticks(range(1, 13))
            ax2.set_xticklabels([MONTH_ABBRS[i] for i in range(1, 13)])
            ax2.grid(True, linestyle='--', alpha=0.6)
            ax2.tick_params(axis='both', which='major', labelsize=15)

            # Set consistent y-axis limits for both plots
            if all_temps:
                y_min, y_max = min(all_temps), max(all_temps)
                y_margin = (y_max - y_min) * 0.1
                ax1.set_ylim(y_min - y_margin, y_max + y_margin)
                ax2.set_ylim(y_min - y_margin, y_max + y_margin)

            # Adjust layout
            plt.tight_layout(rect=[0, 0.03, 1, 0.92])

            # Create directory for saving the output
            figures_dir = os.path.join(city_temporal_dir, 'Figures')
            os.makedirs(figures_dir, exist_ok=True)

            # Save the combined plot (SVG+PDF) through centralized data_io helper
            output_file_base = os.path.join(figures_dir, f"{city_name}_LST_Trends_Combined2")
            data_io.export_figure_set(
                fig,
                output_file_base,
                save_png=False,
                save_svg=True,
                save_pdf=True,
                dpi=DPI,
                bbox_inches='tight'
            )

            print(f"Saved trend plots for {city_name} to {figures_dir}")

            plt.close(fig)  # Close to free memory

        print("\n" + "="*70)
        print("LST TREND PLOTS COMPLETED")
        print("="*70)
        print("Trend line plots have been generated for all cities")
        print("="*70)

    except Exception as e:
        print(f"Error creating trend line plots: {e}")
        traceback_str = traceback.format_exc()
        print(f"Traceback: {traceback_str}")

    print("="*70)
    print("LEGEND GENERATION FOR LST TREND PLOTS")
    print("="*70)

    import matplotlib.patches as mpatches
    from matplotlib.lines import Line2D

    try:
        # Create a standalone legend figure
        legend_fig, legend_ax = plt.subplots(figsize=(12, 3), dpi=300)

        # Hide the axes
        legend_ax.axis('off')

        # Define the legend elements matching your plot styling
        legend_elements = [
            # Day-time elements (orange)
            Line2D([0], [0], color=DAY_COLOR, linewidth=3.75, label='Day-time Median'),
            Line2D([0], [0], color=DAY_COLOR, linewidth=3, linestyle=':', label='Day Trend Line'),
            mpatches.Patch(color=DAY_FILL, label='Day Min-Max Range'),

            # Night-time elements (black/grey)
            Line2D([0], [0], color=NIGHT_COLOR, linewidth=3.75, label='Night-time Median'),
            Line2D([0], [0], color=NIGHT_COLOR, linewidth=3, linestyle=':', label='Night Trend Line'),
            mpatches.Patch(color=NIGHT_FILL, label='Night Min-Max Range'),
        ]

        # Create the legend with horizontal layout
        legend = legend_ax.legend(
            handles=legend_elements,
            loc='center',
            ncol=3,
            fontsize=14,
            frameon=True,
            fancybox=True,
            shadow=True,
            framealpha=0.9,
            edgecolor='black',
            facecolor='white'
        )

        # Adjust legend spacing
        legend.get_frame().set_linewidth(1.5)

        # Adjust the layout to center the legend
        legend_fig.tight_layout()

        # Create directory for legend output in the main stats directory
        legend_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], 'Stats')
        os.makedirs(legend_dir, exist_ok=True)

        # Save the legend (SVG+PDF) through centralized data_io helper
        legend_output_base = os.path.join(legend_dir, "LST_Trend_Plot_Legend")
        data_io.export_figure_set(
            legend_fig,
            legend_output_base,
            save_png=False,
            save_svg=True,
            save_pdf=True,
            dpi=300,
            bbox_inches='tight',
            facecolor='white',
            edgecolor='none'
        )
        legend_output_svg = f"{legend_output_base}.svg"
        legend_output_pdf = f"{legend_output_base}.pdf"

        plt.close(legend_fig)

        print("Legend saved successfully:")
        print(f"  SVG: {legend_output_svg}")
        print(f"  PDF: {legend_output_pdf}")

        # Also create a version for each city's figures directory
        for city_name, aoi_info in aoi_data.items():
            city_figures_dir = os.path.join(PROJECT_CONFIG['lst_output_dir'], f'City_{city_name}',
                                           'LST_Temporal_Analysis', 'Figures')
            if os.path.exists(city_figures_dir):
                city_legend_svg = os.path.join(city_figures_dir, f"{city_name}_LST_Trend_Legend.svg")
                city_legend_pdf = os.path.join(city_figures_dir, f"{city_name}_LST_Trend_Legend.pdf")

                # Copy the legend to each city's directory
                import shutil
                shutil.copy2(legend_output_svg, city_legend_svg)
                shutil.copy2(legend_output_pdf, city_legend_pdf)

                print(f"  Legend copied to {city_name} figures directory")

        print("\n" + "="*70)
        print("LEGEND GENERATION COMPLETED")
        print("="*70)
        print("="*70)

    except Exception as e:
        print(f"Error creating legend: {e}")
        traceback_str = traceback.format_exc()
        print(f"Traceback: {traceback_str}")

In [ ]:
# =============================================================================
# HEATMAP VISUALIZATION: 4 SINGLE-PAGE LST HEATMAPS WITH MANN-KENDALL
# =============================================================================

print("="*70)
print("CREATING 4 SINGLE-PAGE LST HEATMAPS WITH MANN-KENDALL SIGNIFICANCE")
print("="*70)

import seaborn as sns
import pymannkendall as mk

# Import heatmap functions from src (previously defined inline)
from src.visualization import compute_mk_significance, create_single_page_heatmap

# --- A4 dimensions in inches ---
A4_W = 8.27   # A4 portrait width
A4_H = 11.69  # A4 portrait height
YEARLY_FIG_W  = A4_W       # 8.27"
YEARLY_FIG_H  = A4_H / 2   # 5.85"
MONTHLY_FIG_W = A4_H / 2   # 5.85"
MONTHLY_FIG_H = A4_W       # 8.27"

# --- Skip logic: check if all 4 heatmap SVGs already exist ---
_skip_hm = config.SKIP_EXISTING.get('heatmaps', False)
_hm_all_exist = False

if _skip_hm:
    _hm_dir = os.path.join(config.get_output_dir('03_trend_analysis_monthly_annual'), 'Stats')
    _hm_expected = [
        os.path.join(_hm_dir, 'LST_Yearly_Day_Heatmap_MK.svg'),
        os.path.join(_hm_dir, 'LST_Yearly_Night_Heatmap_MK.svg'),
        os.path.join(_hm_dir, 'LST_Monthly_Day_Heatmap_MK.svg'),
        os.path.join(_hm_dir, 'LST_Monthly_Night_Heatmap_MK.svg'),
    ]
    _hm_all_exist = all(os.path.exists(f) for f in _hm_expected)

if _skip_hm and _hm_all_exist:
    print("  SKIPPED all 4 heatmap outputs already exist.")
    print("   Set config.SKIP_EXISTING['heatmaps'] = False to regenerate.")
else:
    # =============== DATA PREPARATION ===============

    lst_times_to_process = []
    if config.LST_CONFIG['lst_time'] in ['day', 'both']:
        lst_times_to_process.append('day')
    if config.LST_CONFIG['lst_time'] in ['night', 'both']:
        lst_times_to_process.append('night')

    combined_stats_file = os.path.join(
        config.get_output_dir('03_trend_analysis_monthly_annual'),
        'Stats', 'All_Cities_LST_Statistics.csv'
    )
    if not os.path.exists(combined_stats_file):
        raise FileNotFoundError(f"Combined statistics file not found: {combined_stats_file}")

    all_stats_df = pd.read_csv(combined_stats_file)

    yearly_heatmaps = {}
    monthly_heatmaps = {}

    for lst_time in lst_times_to_process:
        lst_data = all_stats_df[all_stats_df['day_night'] == lst_time].copy()
        city_yearly_data = {}
        city_monthly_data = {}

        for city_name in aoi_data.keys():
            city_data = lst_data[lst_data['city'] == city_name].copy()
            if not city_data.empty:
                yearly_df = city_data[
                    (city_data['time_period_type'] == 'yearly') &
                    (city_data['is_climatology'] == False)
                ].copy()
                monthly_df = city_data[
                    (city_data['time_period_type'] == 'monthly') &
                    (city_data['is_climatology'] == True)
                ].copy()

                if not yearly_df.empty:
                    yearly_df = yearly_df.rename(columns={'time_period': 'year', 'lst_median': 'median'})
                    yearly_df['year'] = yearly_df['year'].astype(int)
                    city_yearly_data[city_name] = yearly_df
                if not monthly_df.empty:
                    monthly_df = monthly_df.rename(columns={'time_period': 'month', 'lst_median': 'median'})
                    monthly_df['month'] = monthly_df['month'].astype(int)
                    city_monthly_data[city_name] = monthly_df
            else:
                print(f" No statistics data found for {city_name} ({lst_time})")

        if city_yearly_data:
            yearly_heatmaps[lst_time] = temporal_analysis.create_heatmap_data(
                city_yearly_data, time_dim='year', stat='median')
        else:
            yearly_heatmaps[lst_time] = pd.DataFrame()

        if city_monthly_data:
            monthly_heatmaps[lst_time] = temporal_analysis.create_heatmap_data(
                city_monthly_data, time_dim='month', stat='median')
        else:
            monthly_heatmaps[lst_time] = pd.DataFrame()

        print(f" Prepared {lst_time}-time heatmap data")

    # =============== MANN-KENDALL SIGNIFICANCE ===============

    yearly_mk_sig = {}
    monthly_mk_sig = {}
    for lst_time in ['day', 'night']:
        if lst_time in yearly_heatmaps and not yearly_heatmaps[lst_time].empty:
            yearly_mk_sig[lst_time] = compute_mk_significance(yearly_heatmaps[lst_time])
        else:
            yearly_mk_sig[lst_time] = {}
        if lst_time in monthly_heatmaps and not monthly_heatmaps[lst_time].empty:
            monthly_mk_sig[lst_time] = compute_mk_significance(monthly_heatmaps[lst_time])
        else:
            monthly_mk_sig[lst_time] = {}

    for lst_time in ['day', 'night']:
        sig = [c for c, s in yearly_mk_sig.get(lst_time, {}).items() if s]
        tot = len(yearly_mk_sig.get(lst_time, {}))
        print(f"\n{lst_time.title()}-time yearly: {len(sig)}/{tot} cities with significant warming")
        if sig:
            print(f"   {', '.join(sig[:10])}" + (" ..." if len(sig) > 10 else ""))

    # =============== GENERATE 4 HEATMAPS ===============

    output_dir = os.path.join(
        config.get_output_dir('03_trend_analysis_monthly_annual'), 'Stats'
    )
    os.makedirs(output_dir, exist_ok=True)

    # 1. Annual Day-time (vertical half A4)
    if 'day' in yearly_heatmaps and not yearly_heatmaps['day'].empty:
        print("\n1/4  Annual Day-time heatmap...")
        create_single_page_heatmap(
            yearly_heatmaps['day'], yearly_mk_sig['day'],
            lst_time='day', time_dim='year',
            fig_w=YEARLY_FIG_W, fig_h=YEARLY_FIG_H,
            output_path=os.path.join(output_dir, 'LST_Yearly_Day_Heatmap_MK.svg')
        )

    # 2. Annual Night-time (vertical half A4)
    if 'night' in yearly_heatmaps and not yearly_heatmaps['night'].empty:
        print("\n2/4  Annual Night-time heatmap...")
        create_single_page_heatmap(
            yearly_heatmaps['night'], yearly_mk_sig['night'],
            lst_time='night', time_dim='year',
            fig_w=YEARLY_FIG_W, fig_h=YEARLY_FIG_H,
            output_path=os.path.join(output_dir, 'LST_Yearly_Night_Heatmap_MK.svg')
        )

    # 3. Monthly Day-time (horizontal half A4) â€” no MK significance for monthly
    if 'day' in monthly_heatmaps and not monthly_heatmaps['day'].empty:
        print("\n3/4  Monthly Day-time heatmap...")
        create_single_page_heatmap(
            monthly_heatmaps['day'], {},
            lst_time='day', time_dim='month',
            fig_w=MONTHLY_FIG_W, fig_h=MONTHLY_FIG_H,
            output_path=os.path.join(output_dir, 'LST_Monthly_Day_Heatmap_MK.svg')
        )

    # 4. Monthly Night-time (horizontal half A4) â€” no MK significance for monthly
    if 'night' in monthly_heatmaps and not monthly_heatmaps['night'].empty:
        print("\n4/4  Monthly Night-time heatmap...")
        create_single_page_heatmap(
            monthly_heatmaps['night'], {},
            lst_time='night', time_dim='month',
            fig_w=MONTHLY_FIG_W, fig_h=MONTHLY_FIG_H,
            output_path=os.path.join(output_dir, 'LST_Monthly_Night_Heatmap_MK.svg')
        )

    print("\n" + "="*70)
    print("ALL 4 SINGLE-PAGE HEATMAPS COMPLETED")
    print(f"Saved to: {output_dir}")
    print("â˜… = Mann-Kendall significant warming trend (p < 0.05)")
    print("="*70)

## Block 9: Stakeholder Validation Analysis

Validates measured LST trends against stakeholder perceptions:

**Survey Question:**  
*"Have you perceived temperature increases in the last 10 years"*

**Validation Metrics:**

$$\text{Agreement (\%)} = \frac{\text{Stakeholders perceiving warming}}{\text{Total stakeholders}} \times 100$$

**Classification:**
- **Agree**: Perception matches measured trend direction
- **Disagree**: Perception contradicts measured trend
- **Analysis**: Separate validation for day and night LST

**Output Products:**
- Validated heatmaps with perception overlay
- Per-city agreement tables
- Summary statistics on perception accuracy

**Interpretation:**  
High agreement indicates stakeholders accurately perceive thermal changes. Low agreement may reflect:
- Adaptation to gradual warming
- Focus on extreme events vs. mean trends
- Difference between perceived and measured temperature

In [ ]:
# ============================================================================
# BLOCK 9 - ANNUAL & MONTHLY TRENDS
# ============================================================================
# Compare stakeholder perceptions with actual LST trends
# Annual: Warming perception vs yearly LST trends
# Monthly: Peak month identification vs monthly LST hottest months

import numpy as np
import math
from src import data_io

# --- LOAD DATA ONCE FOR BOTH VALIDATIONS ---
survey_file = os.path.join(config.DATA_DIR, 'inputs', 'stakeholder_survey', 'master_composite.csv')
print(f"Loading survey data from: {survey_file}")
survey_df = pd.read_csv(survey_file)

# Filter for WORKSHOP ATTENDEES ONLY
workshop_attendees = survey_df[survey_df['survey status'] == 'Yes'].copy()
workshop_study = workshop_attendees[workshop_attendees['matched_city_y'].notna()].copy()
study_cities = list(aoi_data.keys())

print(f"Total workshop attendees in study cities: {workshop_study['pid'].nunique()}")

# Load LST summary for both annual and monthly peaks
summary_csv_path = os.path.join(main_stats_dir, "City_LST_Enhanced_Summary.csv")
if not os.path.exists(summary_csv_path):
    raise FileNotFoundError(f"Validation requires summary file: {summary_csv_path}")
city_summary = pd.read_csv(summary_csv_path)

# ============================================================================
# ANNUAL VALIDATION: Warming Perception vs Yearly LST Trends
# ============================================================================
print("\n" + "="*70)
print("ANNUAL VALIDATION: Warming Perception vs Yearly LST Trends")
print("="*70)

validation_results = []
total_workshop_attendees = workshop_study['pid'].nunique()

for city_name in study_cities:
    city_responses = workshop_study[workshop_study['matched_city_y'] == city_name]
    n_stakeholders = city_responses['pid'].nunique()

    if n_stakeholders == 0:
        continue

    city_unique = city_responses.groupby('pid').first().reset_index()
    pct_of_total = (n_stakeholders / total_workshop_attendees) * 100
    n_perceived_warming = (city_unique['61_temp_increase_10yrs'] == 'Yes').sum()
    pct_perceived_warming = (n_perceived_warming / n_stakeholders) * 100
    duration_counts = city_unique['56_recency_with_city'].value_counts().to_dict()

    # Get LST trend significance
    city_summary_day = city_summary[(city_summary['City'] == city_name) & (city_summary['Time'] == 'Day')]
    city_summary_night = city_summary[(city_summary['City'] == city_name) & (city_summary['Time'] == 'Night')]

    if not city_summary_day.empty:
        trend_sig_day = city_summary_day['Trend Significance'].values[0]
        is_significant_day = 'significant' in str(trend_sig_day).lower() and 'not significant' not in str(trend_sig_day).lower()
        day_sig_text = 'Significant increase' if is_significant_day else 'No significant trend'
    else:
        is_significant_day = False
        day_sig_text = 'No data'

    if not city_summary_night.empty:
        trend_sig_night = city_summary_night['Trend Significance'].values[0]
        is_significant_night = 'significant' in str(trend_sig_night).lower() and 'not significant' not in str(trend_sig_night).lower()
        night_sig_text = 'Significant increase' if is_significant_night else 'No significant trend'
    else:
        is_significant_night = False
        night_sig_text = 'No data'

    # Agreement: if LST shows significant increase AND stakeholder perceived warming
    if not city_summary_day.empty:
        if is_significant_day:
            overall_agreement = pct_perceived_warming
        else:
            overall_agreement = 0 if pct_perceived_warming > 0 else 100
    else:
        overall_agreement = -1

    validation_results.append({
        'city': city_name,
        'n_stakeholders': n_stakeholders,
        'pct_of_total': pct_of_total,
        'n_perceived_warming': n_perceived_warming,
        'pct_perceived_warming': pct_perceived_warming,
        'overall_agreement': overall_agreement,
        'duration_counts': duration_counts,
        'day_significance': day_sig_text,
        'night_significance': night_sig_text
    })

validation_df = pd.DataFrame(validation_results)
actual_total_stakeholders = validation_df['n_stakeholders'].sum()
validation_df['pct_of_total'] = (validation_df['n_stakeholders'] / actual_total_stakeholders) * 100

print(validation_df[['city', 'n_stakeholders', 'pct_of_total', 'pct_perceived_warming']].to_string(index=False))

# ============================================================================
# MONTHLY VALIDATION: Peak Month Identification vs Monthly LST Hottest Months
# ============================================================================
print("\n" + "="*70)
print("MONTHLY VALIDATION: Peak Month vs LST Hottest Months")
print("="*70)

monthly_validation_results = []
month_map = {'Jan': 'January', 'Feb': 'February', 'Mar': 'March', 'Apr': 'April',
             'May': 'May', 'Jun': 'June', 'Jul': 'July', 'Aug': 'August',
             'Sep': 'September', 'Oct': 'October', 'Nov': 'November', 'Dec': 'December'}

month_ranges = {
    'March – May': ['March', 'April', 'May'],
    'December – February': ['December', 'January', 'February'],
    'June – August': ['June', 'July', 'August'],
    'September – November': ['September', 'October', 'November']
}

# Create normalized lookup for encoding variants
month_ranges_normalized = {}
for key, value in month_ranges.items():
    month_ranges_normalized[key] = value
    alt_key = key.replace('–', ' - ')
    if alt_key != key:
        month_ranges_normalized[alt_key] = value

for city in study_cities:
    city_responses = workshop_study[workshop_study['matched_city_y'] == city].copy()
    if len(city_responses) == 0:
        continue

    n_stakeholders = city_responses['pid'].nunique()
    city_unique = city_responses.groupby('pid').first().reset_index()
    
    city_lst_day = city_summary[(city_summary['City'] == city) & (city_summary['Time'] == 'Day')]
    city_lst_night = city_summary[(city_summary['City'] == city) & (city_summary['Time'] == 'Night')]

    if city_lst_day.empty and city_lst_night.empty:
        continue

    # Extract hottest months
    lst_day_hottest = None
    lst_night_hottest = None

    if not city_lst_day.empty:
        lst_day_abbr = city_lst_day['Hottest Month'].values[0].split(' ')[0]
        lst_day_hottest = month_map.get(lst_day_abbr, lst_day_abbr)

    if not city_lst_night.empty:
        lst_night_abbr = city_lst_night['Hottest Month'].values[0].split(' ')[0]
        lst_night_hottest = month_map.get(lst_night_abbr, lst_night_abbr)

    # Calculate agreement
    peak_month_responses = city_unique['58_temp_peak_city'].dropna()

    if len(peak_month_responses) > 0:
        agree_count = 0
        agree_day_count = 0
        agree_night_count = 0
        agree_both_count = 0

        for response in peak_month_responses:
            response_clean = str(response).strip() if response else None
            
            if response_clean and response_clean in month_ranges_normalized:
                day_agrees = lst_day_hottest in month_ranges_normalized[response_clean] if lst_day_hottest else False
                night_agrees = lst_night_hottest in month_ranges_normalized[response_clean] if lst_night_hottest else False

                if day_agrees or night_agrees:
                    agree_count += 1
                    if day_agrees and night_agrees:
                        agree_both_count += 1
                    elif day_agrees:
                        agree_day_count += 1
                    else:
                        agree_night_count += 1

        overall_agreement = (agree_count / len(peak_month_responses)) * 100
    else:
        overall_agreement = np.nan
        agree_day_count = 0
        agree_night_count = 0
        agree_both_count = 0

    duration_col = '56_recency_with_city'
    duration_counts = city_unique[duration_col].value_counts().to_dict()
    
    # Get the most common peak month response from stakeholders
    most_common_response = None
    if len(peak_month_responses) > 0:
        response_counts = peak_month_responses.value_counts()
        most_common_response = response_counts.idxmax() if len(response_counts) > 0 else None

    monthly_validation_results.append({
        'city': city,
        'n_stakeholders': n_stakeholders,
        'lst_day_hottest': lst_day_hottest if lst_day_hottest else 'N/A',
        'lst_night_hottest': lst_night_hottest if lst_night_hottest else 'N/A',
        'stakeholder_peak_response': most_common_response if most_common_response else 'N/A',
        'overall_agreement': overall_agreement,
        'agree_day_only': agree_day_count,
        'agree_night_only': agree_night_count,
        'agree_both': agree_both_count,
        'duration_counts': duration_counts
    })

monthly_validation_df = pd.DataFrame(monthly_validation_results)
actual_total_monthly = monthly_validation_df['n_stakeholders'].sum()
monthly_validation_df['pct_of_total'] = (monthly_validation_df['n_stakeholders'] / actual_total_monthly) * 100

# ============================================================================
# EXPORT CSVs
# ============================================================================
# Annual CSV (optional, for reference)
annual_export_df = validation_df[['city', 'n_stakeholders', 'pct_of_total', 'pct_perceived_warming']].copy()
annual_summary_path = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'),
    'Stats',
    'Stakeholder_Validation_Annual_Summary.csv'
)
annual_export_df.to_csv(annual_summary_path, index=False)

# Monthly CSV
monthly_export_df = monthly_validation_df[[
    'city', 'n_stakeholders', 'pct_of_total',
    'lst_day_hottest', 'lst_night_hottest',
    'overall_agreement', 'agree_day_only', 'agree_night_only', 'agree_both'
]].copy()
monthly_summary_path = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'),
    'Stats',
    'Stakeholder_Validation_Monthly_Summary.csv'
)
monthly_export_df.to_csv(monthly_summary_path, index=False)

print(f"\n✓ Annual summary CSV: {annual_summary_path}")
print(f"✓ Monthly summary CSV: {monthly_summary_path}")

# ============================================================================
# CREATE BOTH SUNBURST VISUALIZATIONS
# ============================================================================
print("\n" + "="*70)
print("CREATING SUNBURST CHARTS...")
print("="*70)

# Define colors
city_color = '#4a4a4a'
duration_colors_map = {
    'Within the last 5 years': '#FFD700',
    'Between 5 - 10 years': '#1AC9F5',
    'Between 11 - 15 years': '#32CD32',
    'Over 15 years': '#8B008B'
}
agree_color = '#0000FF'
disagree_color = '#FF0000'
duration_categories = ['Within the last 5 years', 'Between 5 - 10 years',
                      'Between 11 - 15 years', 'Over 15 years']

# --- ANNUAL CHART ---
print("\n  Creating Annual Validation Chart...")
total_stakeholders = validation_df['n_stakeholders'].sum()
n_cities = len(validation_df[validation_df['n_stakeholders'] > 0])
max_cols = min(5, n_cities)
n_donut_rows = math.ceil(n_cities / max_cols)
fig_width = 8 * max_cols
fig_height = 8 * n_donut_rows + 3
fig = plt.figure(figsize=(fig_width, fig_height))

height_ratios = [5] * n_donut_rows + [1]
gs = fig.add_gridspec(n_donut_rows + 1, max_cols, height_ratios=height_ratios, hspace=0.3)

city_idx = 0
for idx, city_data in validation_df.iterrows():
    if city_data['n_stakeholders'] == 0:
        continue

    row = city_idx // max_cols
    col = city_idx % max_cols
    ax = fig.add_subplot(gs[row, col])

    n_stakeholders = city_data['n_stakeholders']
    pct_of_total = city_data['pct_of_total']
    overall_agreement = city_data['overall_agreement']
    duration_counts = city_data['duration_counts']

    layer1_sizes = [n_stakeholders, total_stakeholders - n_stakeholders]
    layer1_colors = ['#FFA500', '#E0E0E0']

    agree_pct = overall_agreement
    disagree_pct = 100 - overall_agreement if not np.isnan(overall_agreement) else 0
    layer2_sizes = [agree_pct, disagree_pct] if not np.isnan(overall_agreement) else [0, 0]
    layer2_colors = [agree_color, disagree_color]

    layer3_sizes = []
    layer3_colors = []
    for dur_cat in duration_categories:
        count = duration_counts.get(dur_cat, 0)
        if pd.isna(count):
            count = 0
        if count > 0:
            layer3_sizes.append(float(count))
            layer3_colors.append(duration_colors_map[dur_cat])

    if sum(layer1_sizes) > 0:
        ax.pie(layer1_sizes, radius=0.5, colors=layer1_colors,
               wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2), startangle=90)
    if sum(layer2_sizes) > 0:
        ax.pie(layer2_sizes, radius=0.7, colors=layer2_colors,
               wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2), startangle=90)
    if sum(layer3_sizes) > 0:
        ax.pie(layer3_sizes, radius=0.9, colors=layer3_colors,
               wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2), startangle=90)

    ax.text(0, 0.15, city_data['city'], ha='center', va='center', fontsize=18, fontweight='bold')
    ax.text(0, -0.05, f"{n_stakeholders}", ha='center', va='center', fontsize=32, fontweight='bold')
    ax.text(0, -0.25, f"({pct_of_total:.1f}%)", ha='center', va='center', fontsize=14)

    title_text = f"Day: {city_data['day_significance']}, Night: {city_data['night_significance']}"
    ax.text(0.5, 0.92, title_text, transform=ax.transAxes, fontsize=12, ha='center', va='top',
           bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='gray', linewidth=1))
    ax.set_aspect('equal')
    city_idx += 1

legend_ax = fig.add_subplot(gs[n_donut_rows, :])
legend_ax.axis('off')
legend_elements = [
    plt.Line2D([0], [0], color='w', label='Agreement:', markerfacecolor='none'),
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=agree_color, markersize=20,
               label='  Agrees with LST trend'),
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=disagree_color, markersize=20,
               label='  Disagrees with LST trend'),
    plt.Line2D([0], [0], color='w', label=''),
    plt.Line2D([0], [0], color='w', label='Duration of Stay:', markerfacecolor='none'),
    plt.Line2D([0], [0], color='w', label=''),
]
for dur_cat in duration_categories:
    legend_elements.append(plt.Line2D([0], [0], marker='s', color='w',
                                      markerfacecolor=duration_colors_map[dur_cat], markersize=20,
                                      label=f'  {dur_cat}'))

legend_ax.legend(handles=legend_elements, loc='center', ncol=3, frameon=False, fontsize=14,
                handletextpad=0.5, columnspacing=2)
fig.suptitle(
    f"Annual Validation: Multi-Layer Donuts (Total: {total_stakeholders} stakeholders)\n"
    f"Inner: City Proportion | Middle: Annual Trend Agreement | Outer: Duration",
    fontsize=16, fontweight='bold', y=0.98
)

output_stem = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'),
    'Stats',
    'Stakeholder_Validation_Sunburst_Chart'
)
output_paths = data_io.export_figure_set(fig, output_stem, dpi=300, include_svg=True, include_pdf=True)
plt.close(fig)

print(f"    Saved: {output_paths['svg']}")
print(f"    Saved: {output_paths['pdf']}")

# --- MONTHLY CHART ---
print("\n  Creating Monthly Validation Chart...")
total_monthly_stakeholders = monthly_validation_df['n_stakeholders'].sum()
n_cities_monthly = len(monthly_validation_df[monthly_validation_df['n_stakeholders'] > 0])
max_cols = min(5, n_cities_monthly)
n_donut_rows = math.ceil(n_cities_monthly / max_cols)
fig_width = 8 * max_cols
fig_height = 8 * n_donut_rows + 3
fig_monthly = plt.figure(figsize=(fig_width, fig_height))

height_ratios = [4] * n_donut_rows + [1]
gs_monthly = fig_monthly.add_gridspec(n_donut_rows + 1, max_cols, height_ratios=height_ratios, hspace=0.3)

city_idx = 0
for idx, city_data in monthly_validation_df.iterrows():
    if city_data['n_stakeholders'] == 0:
        continue

    row = city_idx // max_cols
    col = city_idx % max_cols
    ax = fig_monthly.add_subplot(gs_monthly[row, col])

    n_city_stakeholders = city_data['n_stakeholders']
    pct_of_total = city_data['pct_of_total']
    overall_agreement = city_data['overall_agreement']

    layer1_sizes = [n_city_stakeholders, total_monthly_stakeholders - n_city_stakeholders]
    layer1_colors = ['#FFA500', '#E0E0E0']

    agree_pct = overall_agreement
    disagree_pct = 100 - overall_agreement if not np.isnan(overall_agreement) else 0
    layer2_sizes = [agree_pct, disagree_pct] if not np.isnan(overall_agreement) else [0, 0]
    layer2_colors = [agree_color, disagree_color]

    layer3_sizes = []
    layer3_colors = []
    for dur_cat in duration_categories:
        count = city_data['duration_counts'].get(dur_cat, 0)
        if count > 0:
            layer3_sizes.append(count)
            layer3_colors.append(duration_colors_map[dur_cat])

    if sum(layer1_sizes) > 0:
        ax.pie(layer1_sizes, radius=0.5, colors=layer1_colors,
               wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2), startangle=90)
    if sum(layer2_sizes) > 0:
        ax.pie(layer2_sizes, radius=0.7, colors=layer2_colors,
               wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2), startangle=90)
    if sum(layer3_sizes) > 0:
        ax.pie(layer3_sizes, radius=0.9, colors=layer3_colors,
               wedgeprops=dict(width=0.2, edgecolor='white', linewidth=2), startangle=90)

    ax.text(0, 0.15, city_data['city'], ha='center', va='center', fontsize=18, fontweight='bold')
    ax.text(0, -0.05, f"{n_city_stakeholders}", ha='center', va='center', fontsize=32, fontweight='bold')
    ax.text(0, -0.25, f"({pct_of_total:.1f}%)", ha='center', va='center', fontsize=14)

    title_text = f"Day: {city_data['lst_day_hottest']}, Night: {city_data['lst_night_hottest']}"
    ax.text(0.5, 0.92, title_text, transform=ax.transAxes, fontsize=12, ha='center', va='top',
           bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='gray', linewidth=1))
    city_idx += 1

legend_ax_monthly = fig_monthly.add_subplot(gs_monthly[n_donut_rows, :])
legend_ax_monthly.axis('off')

legend_elements_monthly = [
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=agree_color, markersize=15,
               label='  Agreement with Monthly Peak (Day OR Night)'),
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=disagree_color, markersize=15,
               label='  Disagreement with Monthly Peak'),
]
for dur_cat in duration_categories:
    legend_elements_monthly.append(plt.Line2D([0], [0], marker='D', color='w',
                                              markerfacecolor=duration_colors_map[dur_cat], markersize=15,
                                              label=f'  {dur_cat}'))

legend_ax_monthly.legend(handles=legend_elements_monthly, loc='center', frameon=True,
                        fontsize=14, ncol=3, title='Duration of Stay:', title_fontsize=16)

fig_monthly.suptitle(
    f"Monthly Peak Validation: Multi-Layer Split Donuts (Total: {total_monthly_stakeholders} stakeholders)\n"
    f"Inner: City Proportion | Middle: Monthly Peak Agreement (Day OR Night) | Outer: Duration",
    fontsize=16, fontweight='bold', y=0.98
)

monthly_output_stem = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'),
    'Stats',
    'Stakeholder_Validation_Monthly_Sunburst_Chart'
)

monthly_output_paths = data_io.export_figure_set(
    fig_monthly, monthly_output_stem, dpi=300, include_svg=True, include_pdf=True, facecolor='white'
)
plt.close(fig_monthly)

print(f"    Saved: {monthly_output_paths['svg']}")
print(f"    Saved: {monthly_output_paths['pdf']}")

print("\n" + "="*70)
print("STAKEHOLDER VALIDATION COMPLETE")
print("="*70)
print(f"\nOutputs generated:")
print(f"  ✓ Annual & Monthly Charts (SVG + PDF)")
print(f"  ✓ Annual Summary: {annual_summary_path}")
print(f"  ✓ Monthly Summary: {monthly_summary_path}")
print("="*70)


In [ ]:
# [DEPRECATED: Merged into Block 9-10]
# This block has been merged with the annual validation block to avoid repetition
# See merged Block 9-10 above for combined annual and monthly validation logic
pass


In [ ]:
# [DEPRECATED: Merged into Block 9-10]
# Visualization code has been merged into the annual validation block
pass


In [ ]:
# [DEBUG CELL - REMOVED]
pass


---

## Summary & Next Steps

### What We Accomplished
 Extracted monthly and annual LST data for all study cities (2006-2025)  
 Applied hierarchical QA filtering for data quality assurance  
 Computed comprehensive temporal statistics  
 Performed trend analysis with statistical significance testing  
 Generated publication-ready visualizations  
 Validated trends against stakeholder perceptions  

### Key Outputs
- Monthly climatology GeoTIFFs (12 per city)
- Yearly LST GeoTIFFs (20 per city)
- Statistical summaries (CSV files)
- Trend analysis plots (SVG/SVG)
- Cross-city comparison heatmaps
- Stakeholder validation reports

### Continue To
âž¡ï¸ **Notebook 04: Daily LST Analysis**
- High-resolution daily temperature patterns
- Diurnal temperature range (DTR) analysis
- Extreme heat event detection

---

**ðŸ“Â Note:** All outputs from this notebook are stored in `data/outputs/03_trend_analysis_monthly_annual/`

In [ ]:
# ============================================================================
# BLOCK 11: UNIFIED VALIDATION SUMMARY
# ============================================================================
# Combines annual and monthly validation into single comprehensive table

print("\n" + "="*70)
print("CREATING UNIFIED VALIDATION SUMMARY")
print("="*70)

# Merge annual and monthly validation data
unified_validation = validation_df[['city', 'n_stakeholders', 'pct_of_total', 
                                     'n_perceived_warming', 'pct_perceived_warming', 
                                     'day_significance', 'night_significance']].copy()

# Rename for clarity
unified_validation.rename(columns={
    'city': 'City',
    'n_stakeholders': 'N_Stakeholders',
    'pct_of_total': 'Pct_of_Total',
    'n_perceived_warming': 'N_Said_Yes_Warming',
    'pct_perceived_warming': 'Annual_Warming_Agreement(%)',
    'day_significance': 'Day_Trend_Significance',
    'night_significance': 'Night_Trend_Significance'
}, inplace=True)

# Create a display column showing "n/total" format for warming responses
unified_validation['Stakeholder_Warming_Input'] = (
    unified_validation['N_Said_Yes_Warming'].astype(int).astype(str) + '/' + 
    unified_validation['N_Stakeholders'].astype(int).astype(str)
)

# Merge with monthly validation data
monthly_for_merge = monthly_validation_df[['city', 'overall_agreement', 
                                            'lst_day_hottest', 'lst_night_hottest',
                                            'stakeholder_peak_response']].copy()

monthly_for_merge.rename(columns={
    'city': 'City',
    'overall_agreement': 'Monthly_Peak_Agreement(%)',
    'lst_day_hottest': 'Day_Hottest_Month',
    'lst_night_hottest': 'Night_Hottest_Month',
    'stakeholder_peak_response': 'Stakeholder_Peak_Month_Response'
}, inplace=True)

# Merge on City
unified_validation = unified_validation.merge(monthly_for_merge, on='City', how='left')

# Calculate Consensus Level based on both annual and monthly agreement
def calculate_consensus_level(row):
    """
    Determine consensus level based on combined annual and monthly agreement.
    High: Both annual and monthly > 70%
    Medium: One or both 50-70%, or average > 60%
    Low: Either < 50% or low average
    """
    annual_agree = row['Annual_Warming_Agreement(%)']
    monthly_agree = row['Monthly_Peak_Agreement(%)']
    
    if pd.isna(annual_agree) or pd.isna(monthly_agree):
        return 'Unknown'
    
    avg_agreement = (annual_agree + monthly_agree) / 2
    
    if annual_agree > 70 and monthly_agree > 70:
        return 'High'
    elif annual_agree > 50 and monthly_agree > 50 and avg_agreement > 60:
        return 'Medium'
    elif avg_agreement >= 60:
        return 'Medium'
    else:
        return 'Low'

unified_validation['Consensus_Level'] = unified_validation.apply(calculate_consensus_level, axis=1)

# Format agreement percentages to 1 decimal place
unified_validation['Annual_Warming_Agreement(%)'] = unified_validation['Annual_Warming_Agreement(%)'].round(1)
unified_validation['Monthly_Peak_Agreement(%)'] = unified_validation['Monthly_Peak_Agreement(%)'].round(1)
unified_validation['Pct_of_Total'] = unified_validation['Pct_of_Total'].round(1)

# Reorder columns for logical flow
unified_validation = unified_validation[[
    'City', 
    'N_Stakeholders', 
    'Pct_of_Total',
    'Stakeholder_Warming_Input',
    'Annual_Warming_Agreement(%)', 
    'Day_Trend_Significance',
    'Night_Trend_Significance',
    'Stakeholder_Peak_Month_Response',
    'Monthly_Peak_Agreement(%)',
    'Day_Hottest_Month',
    'Night_Hottest_Month',
    'Consensus_Level'
]]

# Remove cities with no stakeholders
unified_validation = unified_validation[unified_validation['N_Stakeholders'] > 0].copy()

# Display the unified table
print("\n" + "="*70)
print("UNIFIED STAKEHOLDER VALIDATION SUMMARY")
print("="*70)
print(unified_validation.to_string(index=False))

# Export to CSV
unified_summary_path = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'),
    'Stats',
    'Stakeholder_Validation_Unified_Summary.csv'
)

unified_validation.to_csv(unified_summary_path, index=False)
print("\n" + "="*70)
print(f"✓ Unified validation summary saved:")
print(f"  {unified_summary_path}")
print("="*70)

# Summary statistics
print("\n" + "="*70)
print("CONSENSUS LEVEL DISTRIBUTION")
print("="*70)

consensus_counts = unified_validation['Consensus_Level'].value_counts()
for level in ['High', 'Medium', 'Low', 'Unknown']:
    count = consensus_counts.get(level, 0)
    if count > 0:
        print(f"{level:10s}: {count:2d} cities ({count/len(unified_validation)*100:5.1f}%)")

avg_annual = unified_validation['Annual_Warming_Agreement(%)'].mean()
avg_monthly = unified_validation['Monthly_Peak_Agreement(%)'].mean()

print(f"\nOverall Agreement Statistics:")
print(f"  Average Annual Warming Agreement:    {avg_annual:6.1f}%")
print(f"  Average Monthly Peak Agreement:      {avg_monthly:6.1f}%")
print(f"  Combined Average:                    {(avg_annual + avg_monthly)/2:6.1f}%")
print("\n" + "="*70)